In [6]:
import argparse
import json
import tensorboard
import tensorboardX
import os
import re
import random
import math
import argparse
import json
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim 
import nni
from nni.nas.nn.pytorch import (
    ModelSpace, LayerChoice, MutableConv2d, MutableBatchNorm2d,
    MutableReLU, MutableLinear, MutableMultiheadAttention,
)
from nni.nas.experiment.config import NasExperimentConfig
from pytorch_lightning import Trainer
from nni.nas.evaluator.pytorch import Lightning, ClassificationModule, Trainer
from nni.nas.experiment import NasExperiment
from nni.nas.space import model_context
from nni.nas.hub.pytorch import DARTS
from nni.nas.strategy import DARTS as DartsStrategy
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchvision import transforms
from torchvision.datasets import CIFAR10
from nni.nas.experiment import NasExperiment
from nni.nas.evaluator import FunctionalEvaluator
import nni.nas.strategy as strategy
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from nni.experiment.config import utils, ExperimentConfig
from pytorch_lightning.callbacks import ModelCheckpoint
torch.set_float32_matmul_precision('medium')
from tqdm import tqdm
from nni.nas.nn.pytorch import LayerChoice, ModelSpace, ValueChoice
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from pytorch_lightning import LightningModule, Trainer
from torchvision import datasets, transforms
from nni.nas.evaluator.pytorch import Classification

# Import the quantized CustomDARTSSpace
import sys
import importlib

from pathlib import Path
# get user name from Path.home() to construct paths

USER = Path.home().name

root = Path.home() / "TESI_FILIPPO_LUCCHESI"
os.chdir(root)
print(f'Current working directory: {Path.cwd()}')

module_dir = root / "code"
if not (module_dir / "DoReFaLayers.py").is_file():
    raise FileNotFoundError(f"DoReFaLayers.py not found in {module_dir}")

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

import DoReFaLayers
importlib.reload(DoReFaLayers)
from DoReFaLayers import MutableQuantConv2d, MutableQuantLinear, dorefa_activation, LSQWeightQuantizer
from nni.nas.oneshot.pytorch.supermodule.operation import (
    MixedConv2d, MixedLinear, MixedMultiHeadAttention,
)

# Reproducibility settings
SEED = 42
os.environ.setdefault('PYTHONHASHSEED', str(SEED))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if hasattr(torch.backends, 'cuda') and hasattr(torch.backends.cuda, 'matmul'):
    torch.backends.cuda.matmul.allow_tf32 = False
if hasattr(torch.backends, 'cudnn'):
    torch.backends.cudnn.allow_tf32 = False
print(f'Using global seed: {SEED}')


class QuantMixedConv2d(MixedConv2d):
    def _get_lsq_quantizer(self, bits: int, per_channel: bool):
        if not hasattr(self, "_lsq_weight_quantizers"):
            self._lsq_weight_quantizers = nn.ModuleDict()
        key = f"{bits}_{'pc' if per_channel else 'pt'}"
        if key not in self._lsq_weight_quantizers:
            self._lsq_weight_quantizers[key] = LSQWeightQuantizer(k=bits, per_channel=per_channel)
        return self._lsq_weight_quantizers[key]

    def forward_with_args(self, in_channels, out_channels, kernel_size, stride, padding, dilation, groups, inputs):
        if isinstance(stride, dict) or isinstance(dilation, dict):
            raise ValueError('Mutable stride/dilation is not supported in differentiable mode.')
        params_mapping = self._freeze_weight_impl(in_channels, out_channels, kernel_size, groups)
        weight, bias, in_channels_per_group = [params_mapping.get(name) for name in ['weight', 'bias', 'in_channels_per_group']]
        num_bits = getattr(self, 'num_bits', None)
        w_quant_bits = num_bits
        if num_bits is not None:
            quantizer = self._get_lsq_quantizer(w_quant_bits, per_channel=True)
            weight = quantizer(weight)
        if isinstance(groups, dict):
            if not isinstance(in_channels_per_group, (int, float)):
                raise ValueError(f'Input channels per group is not numeric: {in_channels_per_group}')
            if inputs.size(1) % in_channels_per_group != 0:
                raise RuntimeError(
                f'Input channels must be divisible by in_channels_per_group, but got input shape {inputs.size()} and '
                f'in_channels_per_group={in_channels_per_group}'
            )
            groups = inputs.size(1) // int(in_channels_per_group)
        if isinstance(kernel_size, dict):
            if isinstance(self.padding, str):
                raise ValueError(f'Use "{self.padding}" in padding is not supported.')
            padding = self.padding
        stride_ = self._to_tuple(stride)
        dilation_ = self._to_tuple(dilation)
        if self.padding_mode != 'zeros':
            return F.conv2d(
                F.pad(inputs, self._reversed_padding_repeated_twice, mode=self.padding_mode),
                weight, bias, stride_, (0, 0), dilation_, groups
            )
        return F.conv2d(inputs, weight, bias, stride_, padding, dilation_, groups)

    @classmethod
    def mutate(cls, module, name, memo, kwargs):
        mixed_op = super().mutate(module, name, memo, kwargs)
        if mixed_op is not None:
            mixed_op.num_bits = getattr(module, 'num_bits', None)
            mixed_op.max_epochs = getattr(module, "max_epochs", None)
            mixed_op.epoch = getattr(module, "epoch", None)
            if mixed_op.num_bits is not None:
                if not hasattr(module, "_lsq_weight_quantizers"):
                    module._lsq_weight_quantizers = nn.ModuleDict()
                mixed_op._lsq_weight_quantizers = module._lsq_weight_quantizers
                for bits in {int(mixed_op.num_bits)}:
                    key = f"{bits}_pc"
                    if key not in module._lsq_weight_quantizers:
                        quantizer = LSQWeightQuantizer(k=bits, per_channel=True)
                        quantizer.init_step_size(module.weight)
                        module._lsq_weight_quantizers[key] = quantizer
        return mixed_op

    def freeze(self, sample):
        arguments = {**self.init_arguments}
        for name, mutable in self.mutable_arguments.items():
            arguments[name] = mutable.freeze(sample)
        operation = self.bound_type(**arguments)
        state_dict = self.freeze_weight(**arguments)
        operation.load_state_dict(state_dict, strict=False)
        return operation


class SafeMixedLinear(MixedLinear):
    def _get_lsq_quantizer(self, bits: int, per_channel: bool):
        if not hasattr(self, "_lsq_weight_quantizers"):
            self._lsq_weight_quantizers = nn.ModuleDict()
        key = f"{bits}_{'pc' if per_channel else 'pt'}"
        if key not in self._lsq_weight_quantizers:
            self._lsq_weight_quantizers[key] = LSQWeightQuantizer(k=bits, per_channel=per_channel)
        return self._lsq_weight_quantizers[key]

    def forward_with_args(self, in_features, out_features, inputs):
        params_mapping = self.freeze_weight(in_features, out_features)
        weight, bias = [params_mapping.get(name) for name in ['weight', 'bias']]
        num_bits = getattr(self, 'num_bits', None)
        w_quant_bits = num_bits
        if num_bits is not None:
            quantizer = self._get_lsq_quantizer(w_quant_bits, per_channel=False)
            weight = quantizer(weight)
        return F.linear(inputs, weight, bias)

    @classmethod
    def mutate(cls, module, name, memo, kwargs):
        mixed_op = super().mutate(module, name, memo, kwargs)
        if mixed_op is not None:
            mixed_op.num_bits = getattr(module, 'num_bits', None)
            mixed_op.max_epochs = getattr(module, "max_epochs", None)
            mixed_op.epoch = getattr(module, "epoch", None)
            if mixed_op.num_bits is not None:
                if not hasattr(module, "_lsq_weight_quantizers"):
                    module._lsq_weight_quantizers = nn.ModuleDict()
                mixed_op._lsq_weight_quantizers = module._lsq_weight_quantizers
                for bits in {int(mixed_op.num_bits)}:
                    key = f"{bits}_pt"
                    if key not in module._lsq_weight_quantizers:
                        print(f"Initializing LSQ quantizer for {bits}-bit weight quantization in SafeMixedLinear.")
                        quantizer = LSQWeightQuantizer(k=bits, per_channel=False)
                        quantizer.init_step_size(module.weight)
                        module._lsq_weight_quantizers[key] = quantizer
        return mixed_op

    def freeze(self, sample):
        arguments = {**self.init_arguments}
        for name, mutable in self.mutable_arguments.items():
            arguments[name] = mutable.freeze(sample)
        operation = self.bound_type(**arguments)
        state_dict = self.freeze_weight(**arguments)
        operation.load_state_dict(state_dict, strict=False)
        return operation

class QuantMixedSelfAttention(MixedMultiHeadAttention):
    """
    Quantized specialization of NNI's MixedMultiHeadAttention.

    This implementation supports standard Q/K/V attention, but is primarily
    intended for self-attention where query, key, and value have the same
    embedding dimension.
    """

    def _get_weight_quantizer(self, name, bits, per_channel=False):
        if not hasattr(self, "_lsq_weight_quantizers"):
            self._lsq_weight_quantizers = nn.ModuleDict()
        key = f"{name}_{bits}_{'pc' if per_channel else 'pt'}"
        if key not in self._lsq_weight_quantizers:
            self._lsq_weight_quantizers[key] = LSQWeightQuantizer(
                k=bits,
                per_channel=per_channel,
            )
        return self._lsq_weight_quantizers[key]

    @classmethod
    def mutate(cls, module, name, memo, mutate_kwargs):
        # MixedMultiHeadAttention.mutate creates:
        #   - the supernet-sized attention module
        #   - mutable argument mappings
        #   - the DARTS sampling policy
        mixed_op = super().mutate(module,name,memo,mutate_kwargs,)
        if mixed_op is None:
            return None
        mixed_op.num_bits = getattr(module, "num_bits", None)
        mixed_op.act_bits = getattr(module,"act_bits", mixed_op.num_bits,)
        if mixed_op.num_bits is not None:
            bits = int(mixed_op.num_bits)
            if not hasattr(module, "_lsq_weight_quantizers"):
                module._lsq_weight_quantizers = nn.ModuleDict()
            mixed_op._lsq_weight_quantizers = (
                module._lsq_weight_quantizers
            )
            # Packed Q/K/V projection.
            if mixed_op.in_proj_weight is not None:
                key = f"in_proj_{bits}_pt"
                if key not in mixed_op._lsq_weight_quantizers:
                    quantizer = LSQWeightQuantizer(
                        k=bits,
                        per_channel=False,
                    )
                    quantizer.init_step_size(
                        mixed_op.in_proj_weight
                    )
                    mixed_op._lsq_weight_quantizers[key] = quantizer
            # Output projection.
            key = f"out_proj_{bits}_pt"
            if key not in mixed_op._lsq_weight_quantizers:
                quantizer = LSQWeightQuantizer(
                    k=bits,
                    per_channel=False,
                )
                quantizer.init_step_size(
                    mixed_op.out_proj.weight
                )
                mixed_op._lsq_weight_quantizers[key] = quantizer
            # Separate Q/K/V weights when kdim or vdim differs.
            for projection_name in [
                "q_proj_weight",
                "k_proj_weight",
                "v_proj_weight",
            ]:
                weight = getattr(
                    mixed_op,
                    projection_name,
                    None,
                )
                if weight is None:
                    continue
                key = f"{projection_name}_{bits}_pt"
                if key not in mixed_op._lsq_weight_quantizers:
                    quantizer = LSQWeightQuantizer(
                        k=bits,
                        per_channel=False,
                    )
                    quantizer.init_step_size(weight)
                    mixed_op._lsq_weight_quantizers[key] = quantizer

        if mixed_op.act_bits is not None:
            bits = int(mixed_op.act_bits)
            mixed_op.act_quants = nn.ModuleDict({
                "query": SignedActQuant(k=bits),
                "key": SignedActQuant(k=bits),
                "value": SignedActQuant(k=bits),
                "q_projected": SignedActQuant(k=bits),
                "k_projected": SignedActQuant(k=bits),
                "v_projected": SignedActQuant(k=bits),
                "scores": SignedActQuant(k=bits),
                # Softmax produces values in [0, 1].
                "probabilities": ActQuant(
                    k=bits,
                    init_step=1.0 / (2**bits - 1),
                ),
                "context": SignedActQuant(k=bits),
                "output": SignedActQuant(k=bits),
            })
        return mixed_op

    def _quantize_weight(self, name, weight):
        if weight is None or self.num_bits is None:
            return weight
        bits = int(self.num_bits)
        key = f"{name}_{bits}_pt"
        return self._lsq_weight_quantizers[key](weight)

    def _quantize_activation(self, name, value):
        if self.act_bits is None:
            return value
        return self.act_quants[name](value)

    @staticmethod
    def _apply_attention_mask(scores, attn_mask):
        if attn_mask is None:
            return scores
        if attn_mask.dtype == torch.bool:
            return scores.masked_fill(
                attn_mask,
                torch.finfo(scores.dtype).min,
            )
        return scores + attn_mask

    @staticmethod
    def _apply_key_padding_mask(scores, key_padding_mask):
        if key_padding_mask is None:
            return scores
        # [B, source_length] -> [B, 1, 1, source_length]
        mask = key_padding_mask[:, None, None, :]
        if mask.dtype == torch.bool:
            return scores.masked_fill(
                mask,
                torch.finfo(scores.dtype).min,
            )
        return scores + mask

    def forward_with_args(
        self,
        # These five arguments must match argument_list from
        # MixedMultiHeadAttention.
        embed_dim,
        num_heads,
        kdim,
        vdim,
        dropout,
        # Runtime forward arguments.
        query,
        key,
        value,
        key_padding_mask=None,
        need_weights=True,
        attn_mask=None,
        average_attn_weights=True,
        is_causal=False,
    ):
        # DARTS does not support differentiable mixtures of num_heads or
        # dropout in NNI's built-in implementation.
        if isinstance(num_heads, dict):
            raise ValueError(
                "Mutable num_heads is not supported by DARTS. "
                "Use a LayerChoice of complete attention candidates."
            )
        if isinstance(dropout, dict):
            raise ValueError(
                "Mutable dropout is not supported by DARTS."
            )
        if is_causal and attn_mask is None:
            target_length = query.size(1 if self.batch_first else 0)
            source_length = key.size(1 if self.batch_first else 0)
            attn_mask = torch.ones(
                target_length,
                source_length,
                device=query.device,
                dtype=torch.bool,
            ).triu(diagonal=1)
        if self.batch_first:
            # Internal representation: [sequence, batch, channels].
            query = query.transpose(0, 1)
            key = key.transpose(0, 1)
            value = value.transpose(0, 1)
        # For a differentiable weighted embed_dim choice, NNI keeps the
        # maximum-sized representation and mixes sliced parameters.
        used_embed_dim = (
            self.embed_dim
            if isinstance(embed_dim, dict)
            else embed_dim
        )
        if used_embed_dim % num_heads != 0:
            raise ValueError(
                f"embed_dim={used_embed_dim} must be divisible by "
                f"num_heads={num_heads}"
            )
        head_dim = used_embed_dim // num_heads
        # Reuse NNI's tested logic for slicing/mixing the supernet weights.
        params = self._freeze_weight_impl(
            embed_dim,
            kdim,
            vdim,
        )
        in_proj_weight = params["in_proj_weight"]
        in_proj_bias = params["in_proj_bias"]

        q_proj_weight = params["q_proj_weight"]
        k_proj_weight = params["k_proj_weight"]
        v_proj_weight = params["v_proj_weight"]

        out_proj_weight = params["out_proj.weight"]
        out_proj_bias = params["out_proj.bias"]

        qkv_same_embed_dim = params["qkv_same_embed_dim"]

        query = self._quantize_activation("query", query)
        key = self._quantize_activation("key", key)
        value = self._quantize_activation("value", value)

        if qkv_same_embed_dim:
            in_proj_weight = self._quantize_weight(
                "in_proj",
                in_proj_weight,
            )

            q_weight, k_weight, v_weight = (
                in_proj_weight.chunk(3, dim=0)
            )

            if in_proj_bias is None:
                q_bias = k_bias = v_bias = None
            else:
                q_bias, k_bias, v_bias = (
                    in_proj_bias.chunk(3, dim=0)
                )

        else:
            q_weight = self._quantize_weight(
                "q_proj_weight",
                q_proj_weight,
            )
            k_weight = self._quantize_weight(
                "k_proj_weight",
                k_proj_weight,
            )
            v_weight = self._quantize_weight(
                "v_proj_weight",
                v_proj_weight,
            )

            if in_proj_bias is None:
                q_bias = k_bias = v_bias = None
            else:
                q_bias, k_bias, v_bias = (
                    in_proj_bias.chunk(3, dim=0)
                )

        q = F.linear(query, q_weight, q_bias)
        k = F.linear(key, k_weight, k_bias)
        v = F.linear(value, v_weight, v_bias)

        q = self._quantize_activation("q_projected", q)
        k = self._quantize_activation("k_projected", k)
        v = self._quantize_activation("v_projected", v)

        target_length, batch_size, _ = q.shape
        source_length = k.size(0)

        # [L, B, C] -> [B, heads, L, head_dim]
        q = (
            q.reshape(
                target_length,
                batch_size,
                num_heads,
                head_dim,
            )
            .permute(1, 2, 0, 3)
        )

        k = (
            k.reshape(
                source_length,
                batch_size,
                num_heads,
                head_dim,
            )
            .permute(1, 2, 0, 3)
        )

        v = (
            v.reshape(
                source_length,
                batch_size,
                num_heads,
                head_dim,
            )
            .permute(1, 2, 0, 3)
        )

        scores = torch.matmul(
            q,
            k.transpose(-2, -1),
        )
        scores = scores / math.sqrt(head_dim)

        # Quantize before applying masks. Quantizing a masked -inf value
        # would turn it into a finite value and break masking.
        scores = self._quantize_activation("scores", scores)

        scores = self._apply_attention_mask(
            scores,
            attn_mask,
        )
        scores = self._apply_key_padding_mask(
            scores,
            key_padding_mask,
        )

        probabilities = torch.softmax(scores, dim=-1)
        probabilities = F.dropout(
            probabilities,
            p=dropout,
            training=self.training,
        )

        probabilities = self._quantize_activation(
            "probabilities",
            probabilities,
        )

        context = torch.matmul(probabilities, v)
        context = self._quantize_activation(
            "context",
            context,
        )

        # [B, heads, L, head_dim] -> [L, B, C]
        context = (
            context.permute(2, 0, 1, 3)
            .contiguous()
            .reshape(
                target_length,
                batch_size,
                used_embed_dim,
            )
        )

        out_proj_weight = self._quantize_weight(
            "out_proj",
            out_proj_weight,
        )

        output = F.linear(
            context,
            out_proj_weight,
            out_proj_bias,
        )

        output = self._quantize_activation(
            "output",
            output,
        )

        if self.batch_first:
            output = output.transpose(0, 1)

        if not need_weights:
            return output, None

        if average_attn_weights:
            returned_weights = probabilities.mean(dim=1)
        else:
            returned_weights = probabilities

        return output, returned_weights

    def freeze(self, sample):
        # Match QuantMixedConv2d/SafeMixedLinear: freeze the sampled NNI
        # constructor arguments and copy the sliced supernet weights.
        arguments = {**self.init_arguments}
        for name, mutable in self.mutable_arguments.items():
            arguments[name] = mutable.freeze(sample)
        operation = self.bound_type(**arguments)
        state_dict = self.freeze_weight(**arguments)
        operation.load_state_dict(state_dict, strict=False)
        return operation


def safe_mutable_conv_hook(module, name, memo, kwargs):
    if isinstance(module, MutableQuantConv2d):
        return QuantMixedConv2d.mutate(module, name, memo, kwargs)
    return False

def safe_mutable_attention_hook(module, name, memo, kwargs):
    if isinstance(module, MutableQuantSelfAttention):
        return QuantMixedSelfAttention.mutate(
            module,
            name,
            memo,
            kwargs,
        )
    return False

def safe_mutable_linear_hook(module, name, memo, kwargs):
    if isinstance(module, MutableQuantLinear):
        return SafeMixedLinear.mutate(module, name, memo, kwargs)
    return False

Current working directory: /home/cc/TESI_FILIPPO_LUCCHESI
Using global seed: 42


In [7]:
class SignedActQuant(nn.Module):
    def __init__(self, k=6, init_step=0.5, init_alpha=6.0):
        super().__init__()

        self.k = k
        self.step_size = nn.Parameter(torch.tensor(float(init_step)))
        self.alpha = nn.Parameter(torch.tensor(float(init_alpha)))

    def forward(self, x):
        step = torch.clamp(self.step_size, min=1e-8)
        alpha = torch.clamp(self.alpha, min=1e-8)

        qn = -(2 ** (self.k - 1))
        qp = 2 ** (self.k - 1) - 1

        grad_scale = 1.0 / (
            (max(abs(qn), abs(qp)) * x.numel()) ** 0.5
        )

        scaled_step = (
            (step - step.detach()) * grad_scale
            + step.detach()
        )

        x = torch.clamp(x, -alpha, alpha)
        x_scaled = torch.clamp(x / scaled_step, qn, qp)

        x_rounded = torch.round(x_scaled)
        x_integer = (
            x_scaled
            + (x_rounded - x_scaled).detach()
        )

        return x_integer * scaled_step

In [8]:
def attach_lsq_step_size_hooks(model, every: int = 1):
    resolved = getattr(model, "_model", getattr(model, "model", model))
    handles = []
    for name, module in resolved.named_modules():
        if isinstance(module, LSQWeightQuantizer):
            def make_hook(module_name, mod):
                def hook(grad):
                    mod._lsq_step_print_count = getattr(mod, "_lsq_step_print_count", 0) + 1
                    if mod._lsq_step_print_count % every == 0:
                        step_preview = mod.step_size.detach().flatten()[:3].tolist()
                        print(
                            f"{module_name}.step_size={step_preview} grad_mean={grad.mean().item():.3e}"
                        )
                return hook
            handles.append(module.step_size.register_hook(make_hook(name, module)))
    print(f"Attached {len(handles)} LSQ step_size hooks.")
    return handles

def remove_lsq_step_size_hooks(handles):
    for handle in handles:
        handle.remove()

# Example usage (run after model/module creation):
# lsq_hooks = attach_lsq_step_size_hooks(module, every=1)
# ... run training step(s) ...
# remove_lsq_step_size_hooks(lsq_hooks)

In [9]:
def lsq_quantizer_uniqueness_report(model):
    resolved = getattr(model, "_model", getattr(model, "model", model))
    quantizers = []
    for module in resolved.modules():
        if isinstance(module, LSQWeightQuantizer):
            quantizers.append(module)
    unique_ids = {id(q) for q in quantizers}
    print(f"LSQ quantizers: {len(quantizers)} total, {len(unique_ids)} unique instances")

    by_owner = {}
    for name, module in resolved.named_modules():
        if hasattr(module, "_lsq_weight_quantizers") and isinstance(module._lsq_weight_quantizers, nn.ModuleDict):
            by_owner[name] = list(module._lsq_weight_quantizers.keys())
    if not by_owner:
        print("No modules with _lsq_weight_quantizers found.")
        return
    print("\nModules that own LSQ quantizers:")
    for name, keys in sorted(by_owner.items()):
        print(f"- {name}: {keys}")

# Example usage (run after model/module creation):
# lsq_quantizer_uniqueness_report(module)

In [10]:
import psutil
import os

p = psutil.Process(os.getpid())

p.nice(19)

# Quantized DARTS Architecture Search

This notebook performs NAS using the quantized CustomDARTSSpace with DoReFa quantization-aware training.

# Data Loading and Augmentation

In [11]:
def cutout_transform(img, length: int = 16):
    h, w = img.size(1), img.size(2)
    mask = np.ones((h, w), np.float32)
    y = np.random.randint(h)
    x = np.random.randint(w)

    y1 = np.clip(y - length // 2, 0, h)
    y2 = np.clip(y + length // 2, 0, h)
    x1 = np.clip(x - length // 2, 0, w)
    x2 = np.clip(x + length // 2, 0, w)

    mask[y1: y2, x1: x2] = 0.
    mask = torch.from_numpy(mask)
    mask = mask.expand_as(img)
    img *= mask
    return img

In [12]:
normalize = transforms.Normalize(
                (0.4914, 0.4822, 0.4465),
                (0.2470, 0.2435, 0.2616)
            ) 

def get_cifar10_dataset(valid: bool = False, train: bool = True, cutout: bool = False):
    if train and not valid:
        transform_list = [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
        ]
        if cutout:
            transform_list.append(cutout_transform)
        transform = transforms.Compose(transform_list)
    else:
        transform = transforms.Compose([
            transforms.ToTensor(), 
        ])

    dataset = nni.trace(CIFAR10)(root='./data', train=train, download=True, transform=transform)
    
    return dataset

def stratified_split_indices(dataset, val_fraction=0.2, seed=42):
    rng = np.random.default_rng(seed)
    labels = np.array(dataset.targets, dtype=int)
    train_indices = []
    val_indices = []
    for label in np.unique(labels):
        label_indices = np.where(labels == label)[0]
        rng.shuffle(label_indices)
        if len(label_indices) <= 1:
            train_indices.extend(label_indices.tolist())
            continue
        val_count = max(1, int(round(len(label_indices) * val_fraction)))
        val_count = min(val_count, len(label_indices) - 1)
        val_indices.extend(label_indices[:val_count].tolist())
        train_indices.extend(label_indices[val_count:].tolist())
    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    return train_indices, val_indices

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)

from torch.utils.data import Subset

batch_size = 128
full_train_data = get_cifar10_dataset(train=True, cutout=True)
test_data = get_cifar10_dataset(train=False)
clean_train_data = get_cifar10_dataset(train=True, cutout=False, valid=True)

train_indices, valid_indices = stratified_split_indices(full_train_data, val_fraction=0.2, seed=SEED)
train_data = Subset(full_train_data, train_indices)
valid_data = Subset(clean_train_data, valid_indices)

print(f"Full training samples: {len(full_train_data)}")
print(f"Training split samples: {len(train_data)}")
print(f"Validation split samples: {len(valid_data)}")
print(f"Test samples: {len(test_data)}")

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    pin_memory=True,
    num_workers=6,
    persistent_workers=True,
    shuffle=True,
    worker_init_fn=seed_worker,
    generator=train_generator,
 )

valid_loader = DataLoader(
    valid_data,
    batch_size=batch_size,
    pin_memory=True,
    num_workers=6,
    persistent_workers=True,
    shuffle=False,
    worker_init_fn=seed_worker,
 )

Full training samples: 50000
Training split samples: 40000
Validation split samples: 10000
Test samples: 10000


# Classification Module

In [13]:
from nni.nas.oneshot.pytorch.supermodule.operation import (
    MixedConv2d,
    MixedLinear,
 )


@nni.trace
class DartsClassificationModule(ClassificationModule):
    def __init__(self, learning_rate: float = 0.001, 
                 weight_decay: float = 0., 
                 auxiliary_loss_weight=None, 
                 max_epochs: int = 600,
                 alpha_tracker=None,
                 auxiliary_exit_weights=None):
        super().__init__(learning_rate=learning_rate, 
                        weight_decay=weight_decay, 
                        export_onnx=False, 
                        num_classes=10)
        self.auxiliary_loss_weight = auxiliary_loss_weight
        self.auxiliary_exit_weights = auxiliary_exit_weights
        self.max_epochs = max_epochs
        self.alpha_tracker = alpha_tracker
        self._last_alpha_capture_epoch = None
        self.alpha_grads = {}
        self.learning_rate = learning_rate
        self._use_aux_exits = (auxiliary_loss_weight is not None) or (auxiliary_exit_weights is not None)
        self._logged_aux_mode = False
    
    def configure_optimizers(self):
        weight_params = []
        arch_params = []

        for name, p in self.named_parameters():
            if not p.requires_grad:
                continue

            if '_arch_alpha' in name:
                arch_params.append(p)
            else:
                weight_params.append(p)

        print(f"Weight params: {len(weight_params)}")
        print(f"Arch params (excluded): {len(arch_params)}")

        optimizer = torch.optim.SGD(
            weight_params,  # ✅ ONLY weights
            lr=self.learning_rate,
            momentum=0.9,
            weight_decay=0.
        )

        return {
            'optimizer': optimizer,
            'lr_scheduler': torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=self.max_epochs, eta_min=0.001
            )
        }   

    def _sync_epoch_to_quant_modules(self, epoch: int) -> None:
        model = getattr(self, '_model', getattr(self, 'model', None))
        if model is None:
            return
        for module in model.modules():
            if hasattr(module, 'num_bits'):
                module.epoch = epoch
                module.max_epochs = self.max_epochs

    def _resolve_outputs(self, outputs):
        if isinstance(outputs, tuple) and len(outputs) == 2:
            y_hat, aux = outputs
            if isinstance(aux, list):
                return y_hat, aux
            return y_hat, [(-1, aux)]
        return outputs, []

    def _compute_aux_exit_loss(self, exit_logits, y):
        if not exit_logits or not self.auxiliary_loss_weight:
            return None
        weights = self.auxiliary_exit_weights
        if weights is None:
            weights = [self.auxiliary_loss_weight] * len(exit_logits)
        total = 0.0
        used = 0
        final_layer_index = None
        model = getattr(self, '_model', getattr(self, 'model', None))
        if model is not None and hasattr(model, 'layers'):
            final_layer_index = len(model.layers)
        for (layer_idx, logits), weight in zip(exit_logits, weights):
            if final_layer_index is not None and layer_idx == final_layer_index:
                continue
            total = total + float(weight) * self.criterion(logits, y)
            used += 1
        if used == 0:
            return None
        return total

    def training_step(self, batch, batch_idx):
        """Training step, customized with auxiliary loss."""
        x, y = batch
        epoch = self.trainer.current_epoch if self.trainer else 0
        self._sync_epoch_to_quant_modules(epoch)
        model = getattr(self, '_model', getattr(self, 'model', None))
        if model is None:
            raise RuntimeError('DartsClassificationModule has no attached model.')
        if self._use_aux_exits:
            outputs = model(x, epoch=epoch, return_all_exits=True)
            y_hat, exit_logits = self._resolve_outputs(outputs)
        else:
            if not self._logged_aux_mode:
                print("Aux exits disabled: running standard forward without exit logits.")
                self._logged_aux_mode = True
            y_hat = model(x, epoch=epoch)
            exit_logits = []
        loss_main = self.criterion(y_hat, y)
        loss = loss_main
        aux_loss = self._compute_aux_exit_loss(exit_logits, y)
        if aux_loss is not None:
            loss = loss + aux_loss
            self.log('train_loss_aux', aux_loss)
        self.log('train_loss_main', loss_main)
        self.log('train_loss', loss, prog_bar=True)
        for name, metric in self.metrics.items():
            self.log('train_' + name, metric(y_hat, y), prog_bar=True)
        #if batch_idx % 100 == 0:  # Log alpha stats every 5 epochs to reduce overhead
            #self.log_alpha_grads()
            #self.log_alphas()
        return loss
    
    def log_alpha_grads(self):
        if not self.alpha_grads:
            print("No alpha gradients recorded yet.")
            return

        print("\n=== Alpha Gradient Stats ===")
        for name, g in self.alpha_grads.items():
            mean = g.mean().item()
            std = g.std().item()
            print(f"{name}: mean={mean:.6e}, std={std:.6e}")
    
    def log_alphas(self):
        print("\n=== Alpha Values ===")
        for name, p in self.named_parameters():
            if 'alpha' in name.lower():
                probs = torch.softmax(p, dim=-1)
                print(f"{name}: {probs.detach().cpu().numpy()}")

    def get_temperature(self, epoch, max_epochs):
        # cosine decay from 5 → 0.1
        import math
        T_max = 5.0
        T_min = 0.1
        return T_min + 0.5 * (T_max - T_min) * (1 + math.cos(math.pi * epoch / max_epochs))

    def apply_alpha_temperature(self, T):
        for name, p in self.named_parameters():
            if 'alpha' in name.lower():
                p.data = p.data / T

    def on_train_epoch_start(self):
        # Logging learning rate at the beginning of every epoch
        self.log('lr', self.trainer.optimizers[0].param_groups[0]['lr'])
        #self.ema = EMA(self.model, decay=0.999)
        #epoch = self.trainer.current_epoch
        #T = self.get_temperature(epoch, self.max_epochs)

        #self.apply_alpha_temperature(T)

        #self.log('temperature', T)
    
    def on_after_backward(self):
        #self.ema.update(self.model)
        return

    def attach_alpha_hooks(self):
        """Register backward hooks to capture gradients of architecture parameters."""
        for name, p in self.named_parameters():
            if p.requires_grad and 'alpha' in name.lower():

                def make_hook(name):
                    def hook(grad):
                        self.alpha_grads[name] = grad.detach().clone()
                    return hook

                p.register_hook(make_hook(name))
                
    def _verify_custom_quant_runtime(self):
        model = getattr(self, '_model', getattr(self, 'model', None))
        if model is None:
            raise RuntimeError('Cannot verify quantized runtime without an attached model.')

        modules = list(model.modules())
        quant_convs = sum(isinstance(m, QuantMixedConv2d) for m in modules)
        quant_linears = sum(isinstance(m, SafeMixedLinear) for m in modules)
        quant_attentions = sum(
            isinstance(m, QuantMixedSelfAttention) for m in modules
        )

        leftovers = [
            type(m).__name__
            for m in modules
            if isinstance(
                m,
                (
                    MutableQuantConv2d,
                    MutableQuantLinear,
                    MutableQuantSelfAttention,
                ),
            )
        ]

        if leftovers:
            raise RuntimeError(
                'Custom mutable modules survived NNI mutation: '
                + ', '.join(sorted(set(leftovers)))
            )
        if quant_convs == 0 or quant_linears == 0:
            raise RuntimeError(
                'Quantized convolution/linear hooks were not applied.'
            )
        if quant_attentions == 0:
            raise RuntimeError(
                'Attention candidates exist, but QuantMixedSelfAttention '
                'was not installed by the mutation hook.'
            )

        print(
            'Verified custom NNI runtime modules: '
            f'{quant_convs} QuantMixedConv2d, '
            f'{quant_linears} SafeMixedLinear, '
            f'{quant_attentions} QuantMixedSelfAttention.'
        )

    def on_train_start(self):
        self._verify_custom_quant_runtime()
        self.attach_alpha_hooks()
        print("\n=== Attached alpha gradient hooks ===\n")

    def _capture_alphas_once_per_epoch(self, hook_name: str):
        if self.alpha_tracker is None:
            return

        epoch = int(self.trainer.current_epoch)
        if self._last_alpha_capture_epoch == epoch:
            return

        try:
            self.alpha_tracker.capture_alphas(self, epoch)
            self._last_alpha_capture_epoch = epoch
            # print(f"AlphaTracker: captured at epoch {epoch} from {hook_name}")
        except Exception as e:
            print(f"Alpha capture failed at epoch {epoch} during {hook_name}: {e}")

    def on_train_epoch_end(self):
        self._capture_alphas_once_per_epoch("epoch_end")

    def on_validation_epoch_end(self):
        self._capture_alphas_once_per_epoch("on_validation_epoch_end")



    

In [14]:
class DepthwiseSeparableBlock(nn.Module):
    """
    MobileNet-style depthwise separable block.
    """

    def __init__(self, in_channels, out_channels, stride, num_bits):
        super().__init__()

        self.block = nn.Sequential(
            MutableQuantConv2d(
                in_channels,
                in_channels,
                kernel_size=3,
                padding=1,
                stride = stride,
                groups=in_channels,
                bias=False,
                num_bits=num_bits,
            ),

            MutableBatchNorm2d(in_channels),
            MutableReLU(),

            # pointwise
            MutableQuantConv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False,
                num_bits=num_bits,
            ),

            MutableBatchNorm2d(out_channels),
            MutableReLU(),
        )

    def forward(self, x):
        return self.block(x)

In [15]:
class ActQuant(nn.Module):
    def __init__(self, k=6, init_step=0.5):
        super().__init__()

        self.k = k

        # learned quantization step size
        self.step_size = nn.Parameter(torch.tensor(init_step))

        self.alpha = nn.Parameter(torch.tensor(6.0))

    def forward(self, x):

        # positive step size
        s = torch.clamp(self.step_size, min=1e-8)

        Qn = 0 # -(2 ** (self.k - 1))
        Qp = (2 ** self.k) - 1

        # scale gradient like LSQ paper
        grad_scale = 1.0 / ((Qp * x.numel()) ** 0.5)

        s_scale = (s - s.detach()) * grad_scale + s.detach()

        alpha = torch.clamp(self.alpha, min=1e-8)
        x = torch.clamp(x, -alpha, alpha)
        # quantize
        x_div = x / s_scale
        x_clamped = torch.clamp(x_div, Qn, Qp)

        x_q = torch.round(x_clamped)

        # STE
        x_q = x_clamped + (x_q - x_clamped).detach()

        return x_q * s_scale

class MutableQuantSelfAttention(MutableMultiheadAttention):
    """NNI mutable MHA carrying fixed quantization metadata.

    num_bits and act_bits are deliberately removed from trace_kwargs:
    they are not architecture choices.  The attention mutation hook copies
    them to QuantMixedSelfAttention, just like the convolution/linear hooks.
    """

    def __init__(
        self,
        embed_dim,
        num_heads,
        dropout=0.0,
        bias=True,
        add_bias_kv=False,
        add_zero_attn=False,
        kdim=None,
        vdim=None,
        batch_first=False,
        device=None,
        dtype=None,
        num_bits=6,
        act_bits=None,
    ):
        super().__init__(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            bias=bias,
            add_bias_kv=add_bias_kv,
            add_zero_attn=add_zero_attn,
            kdim=kdim,
            vdim=vdim,
            batch_first=batch_first,
            device=device,
            dtype=dtype,
        )
        self.num_bits = int(num_bits)
        self.act_bits = int(num_bits if act_bits is None else act_bits)
        self.trace_kwargs.pop('num_bits', None)
        self.trace_kwargs.pop('act_bits', None)


class SpatialSelfAttentionBlock(nn.Module):
    """Adapt NCHW CNN features to NNI multi-head self-attention."""

    def __init__(self, in_channels, out_channels, num_heads, num_bits):
        super().__init__()
        if out_channels % num_heads != 0:
            raise ValueError(
                f'out_channels={out_channels} must be divisible by '
                f'num_heads={num_heads}'
            )

        if in_channels == out_channels:
            self.input_projection = nn.Identity()
        else:
            self.input_projection = nn.Sequential(
                MutableQuantConv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=1,
                    padding=0,
                    bias=False,
                    num_bits=num_bits,
                ),
                MutableBatchNorm2d(out_channels),
                MutableReLU(),
            )

        # Pre-normalization produces signed values. QuantMixedSelfAttention
        # immediately applies its signed query/key/value quantizers.
        self.pre_norm = nn.LayerNorm(out_channels)
        self.attention = MutableQuantSelfAttention(
            embed_dim=out_channels,
            num_heads=num_heads,
            dropout=0.0,
            bias=False,
            batch_first=True,
            num_bits=num_bits,
            act_bits=num_bits,
        )
        self.output_bn = MutableBatchNorm2d(out_channels)
        self.output_relu = MutableReLU()

    def forward(self, x):
        x = self.input_projection(x)
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        tokens = self.pre_norm(tokens)
        tokens, _ = self.attention(
            tokens,
            tokens,
            tokens,
            need_weights=False,
        )
        x = tokens.transpose(1, 2).reshape(
            batch,
            channels,
            height,
            width,
        )
        x = self.output_bn(x)
        return self.output_relu(x)


class ResidualBlock(nn.Module):
    def __init__(self, op: nn.Module, in_channels: int, out_channels: int, num_bits: int):
        super().__init__()
        self.op = op
        if in_channels != out_channels:
            self.proj = nn.Sequential(
                MutableQuantConv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=1,
                    bias=False,
                    num_bits=num_bits,
                ),
                MutableBatchNorm2d(out_channels),
            )
        else:
            self.proj = nn.Identity()
        self.post_relu = MutableReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.op(x)
        res = self.proj(x)
        #print(f"Residual block: op output shape {out.shape}, residual shape {res.shape}")
        return self.post_relu(out + res)


class SkipConnect(nn.Module):
    def __init__(self, in_channels, out_channels, num_bits):
        super().__init__()
        if in_channels == out_channels:
            self.op = nn.Identity()
        else:
            # Cheap 1x1 projection to match channel dimensions
            self.op = nn.Sequential(
                MutableQuantConv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=1,
                    padding=0,
                    bias=False,
                    num_bits=num_bits,
                ),
                MutableBatchNorm2d(out_channels)
            )

    def forward(self, x):
        return self.op(x)

class CustomDARTSSpaceResidual(ModelSpace):
    """CustomDARTSSpace with always-on residuals inside each block."""

    def __init__(
        self,
        input_channels: int = 3,
        channels: int = 64,
        num_classes: int = 10,
        layers: int = 7,
        verbose: int = 0,
        drop_path_prob: float = 0.1,
        num_bits: int = 4,
        max_epochs: int = 200,
    ):
        super().__init__()
        self.layers = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        self.drop_path_prob = drop_path_prob
        self.verbose = verbose
        self.max_epochs = max_epochs
        self.num_bits = num_bits
        self.downsample_indices = {1, 3, 6}

        layer0_out = 32
        layer1_out = 32
        layer2_out = 64
        layer3_out = 64
        layer4_out = 128
        layer5_out = 128
        layer6_out = 256
        layer7_out = 200

        self.preliminary_layer = MutableQuantConv2d(
            input_channels,
            layer0_out,
            kernel_size=3,
            padding=1,
            bias=False,
            num_bits=num_bits,
        )
        self.layer0_bn = MutableBatchNorm2d(layer0_out)
        self.layer0_relu = MutableReLU()


        def _downsample_choice(channels, label):
            return LayerChoice(
                [
                    nn.AvgPool2d(kernel_size=2, stride=2),
                    nn.Sequential(
                        MutableQuantConv2d(
                            channels,
                            channels,
                            kernel_size=1,
                            stride=2,
                            bias=False,
                            num_bits=num_bits,
                        ),
                        MutableBatchNorm2d(channels),
                    ),
                ],
                label=label,
            )

        def _residual_layer(op_choice: nn.Module, in_ch: int, out_ch: int):
            return ResidualBlock(op_choice, in_ch, out_ch, num_bits)

        def _attention_candidate(in_ch: int, out_ch: int):
            # Attention is offered only once features are 8x8. DARTS evaluates
            # every LayerChoice candidate, so using global attention at 32x32
            # or 16x16 would make the score tensor unnecessarily large.
            return SpatialSelfAttentionBlock(
                in_channels=in_ch,
                out_channels=out_ch,
                num_heads=8,
                num_bits=num_bits,
            )

        layer1_op = LayerChoice(
            [

                nn.Sequential(
                    MutableQuantConv2d(
                        layer0_out,
                        layer1_out,
                        kernel_size=3,
                        stride=1,
                        padding=1,
                        bias=False,
                        num_bits=num_bits,
                    ),
                    MutableBatchNorm2d(layer1_out),
                    MutableReLU(),
                ),
                DepthwiseSeparableBlock(
                    layer0_out,
                    layer1_out,
                    stride=1,
                    num_bits=num_bits,
                ),
                #SkipConnect(layer0_out, layer1_out, num_bits),
            ],
            label="layer_1_op",
        )
        self.layers.append(_residual_layer(layer1_op, layer0_out, layer1_out))
        self.downsamples.append(nn.Identity())

        layer2_op = LayerChoice(
            [
                nn.Sequential(
                    MutableQuantConv2d(
                        layer1_out,
                        layer2_out,
                        kernel_size=3,
                        stride=1,
                        padding=1,
                        bias=False,
                        num_bits=num_bits,
                    ),
                    MutableBatchNorm2d(layer2_out),
                    MutableReLU(),
                ),
                DepthwiseSeparableBlock(
                    layer1_out,
                    layer2_out,
                    stride=1,
                    num_bits=num_bits,
                ),
            ],
            label="layer_2_op",
        )
        self.layers.append(_residual_layer(layer2_op, layer1_out, layer2_out))
        self.downsamples.append(_downsample_choice(layer2_out, "layer_2_down"))

        layer3_op = LayerChoice(
            [
                nn.Sequential(
                    MutableQuantConv2d(
                        layer2_out,
                        layer3_out,
                        kernel_size=3,
                        stride=1,
                        padding=1,
                        bias=False,
                        num_bits=num_bits,
                    ),
                    MutableBatchNorm2d(layer3_out),
                    MutableReLU(),
                ),
                DepthwiseSeparableBlock(
                    layer2_out,
                    layer3_out,
                    stride=1,
                    num_bits=num_bits,
                ),
                #SkipConnect(layer2_out, layer3_out, num_bits),
            ],
            label="layer_3_op",
        )
        self.layers.append(_residual_layer(layer3_op, layer2_out, layer3_out))
        self.downsamples.append(nn.Identity())

        layer4_op = LayerChoice(
            [
                nn.Sequential(
                    MutableQuantConv2d(
                        layer3_out,
                        layer4_out,
                        kernel_size=3,
                        stride=1,
                        padding=1,
                        bias=False,
                        num_bits=num_bits,
                    ),
                    MutableBatchNorm2d(layer4_out),
                    MutableReLU(),
                ),
                DepthwiseSeparableBlock(
                    layer3_out,
                    layer4_out,
                    stride=1,
                    num_bits=num_bits,
                ),
            ],
            label="layer_4_op",
        )
        self.layers.append(_residual_layer(layer4_op, layer3_out, layer4_out))
        self.downsamples.append(_downsample_choice(layer4_out, "layer_4_down"))

        layer5_op = LayerChoice(
            [
                nn.Sequential(
                    MutableQuantConv2d(
                        layer4_out,
                        layer5_out,
                        kernel_size=3,
                        stride=1,
                        padding=1,
                        bias=False,
                        num_bits=num_bits,
                    ),
                    MutableBatchNorm2d(layer5_out),
                    MutableReLU(),
                ),
                DepthwiseSeparableBlock(
                    layer4_out,
                    layer5_out,
                    stride=1,
                    num_bits=num_bits,
                ),
                _attention_candidate(layer4_out, layer5_out),
                #SkipConnect(layer4_out, layer5_out, num_bits),
            ],
            label="layer_5_op",
        )
        self.layers.append(_residual_layer(layer5_op, layer4_out, layer5_out))
        self.downsamples.append(nn.Identity())

        layer6_op = LayerChoice(
            [
                nn.Sequential(
                    MutableQuantConv2d(
                        layer5_out,
                        layer6_out,
                        kernel_size=3,
                        stride=1,
                        padding=1,
                        bias=False,
                        num_bits=num_bits,
                    ),
                    MutableBatchNorm2d(layer6_out),
                    MutableReLU(),
                ),
                DepthwiseSeparableBlock(
                    layer5_out,
                    layer6_out,
                    stride=1,
                    num_bits=num_bits,
                ),
                _attention_candidate(layer5_out, layer6_out),
                #SkipConnect(layer5_out, layer6_out, num_bits),
            ],
            label="layer_6_op",
        )
        self.layers.append(_residual_layer(layer6_op, layer5_out, layer6_out))
        self.downsamples.append(nn.Identity())

        layer7_op = LayerChoice(
            [
                nn.Sequential(
                    MutableQuantConv2d(
                        layer6_out,
                        layer7_out,
                        kernel_size=3,
                        stride=1,
                        padding=1,
                        bias=False,
                        num_bits=num_bits,
                    ),
                    MutableBatchNorm2d(layer7_out),
                    MutableReLU(),
                ),
                DepthwiseSeparableBlock(
                    layer6_out,
                    layer7_out,
                    stride=1,
                    num_bits=num_bits,
                ),
                _attention_candidate(layer6_out, layer7_out),
            ],
            label="layer_7_op",
        )
        self.layers.append(_residual_layer(layer7_op, layer6_out, layer7_out))
        self.downsamples.append(_downsample_choice(layer7_out, "layer_7_down"))

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = MutableQuantLinear(layer7_out, num_classes, num_bits=num_bits)
        self.epoch = 0

        self.actQuantPre = ActQuant(k=num_bits)
        self.actQuantPost = ActQuant(k=num_bits)
        self.actQuants = nn.ModuleList([
            ActQuant(k=num_bits) for _ in range(layers)
        ])

    def init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor, epoch: int = 0) -> torch.Tensor:
        self.epoch = epoch
        act_quant_bits = self.num_bits # if epoch < self.max_epochs // 3 else self.num_bits
        #if epoch > self.max_epochs // 3 and epoch < 2 * self.max_epochs // 3:
            #act_quant_bits2 = 8
            #x = dorefa_activation(x, act_quant_bits2)
        #elif epoch >= 2 * self.max_epochs // 3:
        act_quant_bits2 = self.num_bits # 6
        x = self.actQuantPre(x)
        x = self.preliminary_layer(x)
        x = self.layer0_bn(x)
        x = self.layer0_relu(x)
        if self.verbose == 1:
            print(f"After preliminary layer: {x.shape}")

        for i, layer in enumerate(self.layers):
            x = self.actQuants[i](x)
            x = layer(x)
            if self.verbose == 1:
                print(f"After layer {i + 1}: {x.shape}")
            if i in self.downsample_indices:
                downsample = self.downsamples[i]
                x = downsample(x)
                if self.verbose == 1:
                    print(f"After downsample choice {i + 1}: {x.shape}")

        x = self.pool(x)
        x = torch.flatten(x, 1)
        # if epoch > self.max_epochs // 3 and epoch < 2 * self.max_epochs // 3:
        #     act_quant_bits2 = 8
        #     x = dorefa_activation(x, act_quant_bits2)
        # elif epoch >= 2 * self.max_epochs // 3:
        act_quant_bits2 = self.num_bits # 6
        x = self.actQuantPost(x)
        x = self.classifier(x)
        if self.verbose == 1:
            print(f"After classifier: {x.shape}")

        return x

    def set_drop_path_prob(self, drop_path_prob: float):
        self.drop_path_prob = drop_path_prob
        for layer in self.layers:
            if hasattr(layer, "set_drop_path_prob"):
                layer.set_drop_path_prob(drop_path_prob)


# Optional: keep the original name for downstream cells without edits.
CustomDARTSSpace = CustomDARTSSpaceResidual

# Evaluator Setup

Checkpoint callback and Lightning Evaluator configuration

In [16]:
# Checkpoint callback
checkpoint_callback = ModelCheckpoint(
    monitor='train_acc', 
    dirpath='./checkpoints',
    filename='best-checkpoint',
    save_top_k=1,
    mode='max'
)

In [17]:
# Alpha Evolution Tracking and Visualization
# This cell prepares infrastructure to track architecture parameter (alpha) evolution during training

import os
import re
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns


def _next_versioned_path(folder: str, base_name: str, ext: str) -> str:
    os.makedirs(folder, exist_ok=True)
    pattern = re.compile(rf"^{re.escape(base_name)}_v(\d+){re.escape(ext)}$")
    max_v = 0
    for fname in os.listdir(folder):
        match = pattern.match(fname)
        if match:
            max_v = max(max_v, int(match.group(1)))
    return os.path.join(folder, f"{base_name}_v{max_v + 1}{ext}")


class AlphaTracker:
    """Track architecture parameters (alphas) throughout training and persist them to disk."""
    
    def __init__(self, save_path: str | None = None):
        self.history = []
        self.choice_names = []
        self.num_choices_per_layer = {}
        self.save_path = save_path
    
    def _resolve_model(self, model):
        return getattr(model, '_model', getattr(model, 'model', model))

    def _save_history(self):
        if self.save_path:
            os.makedirs(os.path.dirname(self.save_path) or '.', exist_ok=True)
            torch.save(self.history, self.save_path)
    
    def capture_alphas(self, model, epoch):
        """Capture current alpha values from the model."""
        resolved_model = self._resolve_model(model)
        alphas_snapshot = {}
        
        # Extract all arch_alpha tensors from the model
        for name, param in resolved_model.named_parameters():
            if '_arch_alpha' in name:
                # name format: layers.0._arch_alpha or layers.0.1._arch_alpha.feature1
                alphas_snapshot[name] = param.detach().cpu().clone()
                
                # Extract the choice name if it exists
                if '._arch_alpha.' in name:
                    choice_name = name.split('._arch_alpha.')[1]
                else:
                    # Extract from module path
                    parts = name.split('.')
                    layer_idx = parts[1] if len(parts) > 1 else 'unknown'
                    choice_name = f'layer_{int(layer_idx)+1}'
                
                if choice_name not in self.choice_names:
                    self.choice_names.append(choice_name)
                    self.num_choices_per_layer[choice_name] = alphas_snapshot[name].shape[0]
        
        # Store epoch data only if we actually found alpha tensors
        if alphas_snapshot:
            self.history.append({
                'epoch': epoch,
                'alphas': alphas_snapshot
            })
            self._save_history()
        else:
            print(f"AlphaTracker: no _arch_alpha tensors found at epoch {epoch}; resolved model type = {type(resolved_model).__name__}")
    
    def get_alpha_statistics(self):
        """Compute statistics about alpha evolution."""
        if not self.history:
            return None
        
        stats = {}
        for choice_name in self.choice_names:
            stats[choice_name] = {
                'initial': [],
                'final': [],
                'entropy_evolution': [],
                'max_prob_evolution': [],
                'winning_choice_evolution': []
            }
        
        # Iterate through epochs
        for epoch_data in self.history:
            alphas = epoch_data['alphas']
            
            for name, alpha_tensor in alphas.items():
                if '._arch_alpha.' in name:
                    choice_name = name.split('._arch_alpha.')[1]
                else:
                    parts = name.split('.')
                    choice_name = f'layer_{int(parts[1])+1}' if len(parts) > 1 else 'unknown'
                
                if choice_name in stats:
                    # Compute softmax to get probabilities
                    probs = torch.softmax(alpha_tensor, dim=0).numpy()
                    
                    # Entropy
                    entropy = -np.sum(probs * np.log(probs + 1e-10))
                    stats[choice_name]['entropy_evolution'].append(entropy)
                    
                    # Max probability (polarization measure)
                    max_prob = np.max(probs)
                    stats[choice_name]['max_prob_evolution'].append(max_prob)
                    
                    # Winning choice
                    winning_idx = np.argmax(probs)
                    stats[choice_name]['winning_choice_evolution'].append(winning_idx)
        
        return stats


# Save alpha history to a versioned, workspace-local file so the NNI trial process can write to it.
alpha_history_dir = '/home/filippo/TESI_FILIPPO_LUCCHESI/alpha_histories'
alpha_history_path = _next_versioned_path(alpha_history_dir, 'alpha_history_6bit', '.pt')

# Initialize tracker
alpha_tracker = AlphaTracker(save_path=alpha_history_path)
print(f"AlphaTracker initialized. It will save alpha history to: {alpha_history_path}")


AlphaTracker initialized. It will save alpha history to: /home/filippo/TESI_FILIPPO_LUCCHESI/alpha_histories/alpha_history_6bit_v5.pt


# NAS Search with Quantized Model

Using CustomDARTSSpace with DoReFa quantization

In [18]:
def get_next_experiment_name(experiment_working_directory: str, base_name: str = "PhotonicDARTS_Quant-v"):
    """Generate the next experiment name by incrementing the version number."""
    if not os.path.exists(experiment_working_directory):
        os.makedirs(experiment_working_directory)
    
    dirs = os.listdir(experiment_working_directory)
    pattern = re.compile(rf'{base_name}(\d+)')

    highest_i = 0
    for d in dirs:
        match = pattern.match(d)
        if match:
            i_value = int(match.group(1))
            if i_value > highest_i:
                highest_i = i_value
    return f"{base_name}{highest_i + 1}"

In [19]:
strategy = DartsStrategy(
    arc_learning_rate=1e-3,
    gradient_clip_val=1,
    mutation_hooks=[
        safe_mutable_attention_hook,
        safe_mutable_conv_hook,
        safe_mutable_linear_hook,
    ],
)

max_epochs = 400

num_bits = 6



def search(
    log_dir: str,
    batch_size: int = 128,
    num_bits: int = 4,
    run_name: str | None = None,
):
    """
    Run DARTS architecture search with quantized model.
    
    Parameters:
    -----------
    log_dir : str
        Directory for logging
    batch_size : int
        Batch size for training
    num_bits : int
        Number of bits for DoReFa quantization (default: 4)
    run_name : str | None
        Explicit TensorBoard/NNI run name. If omitted, a meaningful versioned
        name such as PhotonicDARTS_Quant-v3 is generated.
    
    Returns:
    --------
    NasExperiment
        The completed NAS experiment with search results
    """
    
    # Define model search space with quantization
    model_space = CustomDARTSSpaceResidual(
        input_channels=3, 
        channels=64, 
        num_classes=10, 
        layers=7, 
        verbose=0,
        num_bits=num_bits,  # Quantization parameter
        max_epochs=max_epochs,
    )
    model_space.init_weights()
    model_space.set_drop_path_prob(0.)

    


    # Use one readable identifier for the NNI experiment and TensorBoard.
    experiment_working_directory = "./DartsCheckpoints_Quant"
    if run_name is None:
        run_name = get_next_experiment_name(
            os.path.join(log_dir, "PhotonicDARTS_Quant")
        )

    run_evaluator = build_evaluator(
        log_dir=log_dir,
        run_name=run_name,
    )

    # Run NAS experiment
    exp_config = NasExperimentConfig.default(model_space, run_evaluator, strategy)
    exp_config.experiment_working_directory = experiment_working_directory
    exp_config.experiment_name = run_name
    exp_config.trial_concurrency = 1
    
    experiment = NasExperiment(
        model_space,
        run_evaluator,
        strategy,
        config=exp_config,
    )
    experiment.run()

    lsq_quantizer_uniqueness_report(model_space)
    

    return experiment

In [20]:
# Inspect DARTS alpha optimizer defaults and runtime configuration
import inspect
from nni.nas.strategy import DARTS as DartsStrategy

print("DartsStrategy.__init__ signature:")
print(inspect.signature(DartsStrategy.__init__))

if 'strategy' in globals():
    print("\nStrategy instance:", type(strategy))
    for attr in [
        "arc_learning_rate",
        "arc_weight_decay",
        "arc_optimizer",
        "alpha_optimizer",
        "arch_optimizer",
        "arch_optimizer_name",
        "arch_lr",
    ]:
        if hasattr(strategy, attr):
            print(f"{attr} =", getattr(strategy, attr))
    # Show instantiated arch optimizer if present
    arch_opt = None
    for attr in ["_arch_optimizer", "arch_optimizer", "alpha_optimizer"]:
        if hasattr(strategy, attr):
            val = getattr(strategy, attr)
            if val is not None and "Optimizer" in type(val).__name__:
                arch_opt = val
                break
    if arch_opt is not None:
        print("\nArch optimizer instance:", arch_opt)
        print("Arch optimizer defaults:", arch_opt.defaults)
    else:
        print("\nNo arch optimizer instance found on strategy (may be created lazily during run).")
else:
    print("strategy is not defined; run the cell that creates DartsStrategy first.")

DartsStrategy.__init__ signature:
(self, *, arc_learning_rate: 'float' = 0.0003, gradient_clip_val: 'float | None' = None, log_prob_every_n_step: 'int' = 10, warmup_epochs: 'int' = 0, penalty: 'dict | ExpectationProfilerPenalty | None' = None, **kwargs)

Strategy instance: <class 'nni.nas.oneshot.pytorch.strategy.DARTS'>
arc_learning_rate = 0.001

No arch optimizer instance found on strategy (may be created lazily during run).


In [21]:
if 'strategy' in globals():
    if hasattr(strategy, 'arc_learning_rate'):
        print('arch_lr =', strategy.arc_learning_rate)
    else:
        print('arch_lr not set on strategy; inspect DartsStrategy signature or NNI defaults')
else:
    print('strategy is not defined in this session; run the cell that creates DartsStrategy first')

arch_lr = 0.001


In [22]:
# Build a fresh evaluator and logger for each search run.
# Passing a string as TensorBoardLogger.version prevents Lightning from
# generating the generic version_n directory name.

from nni.nas.evaluator.pytorch import Lightning, Trainer


def build_evaluator(log_dir: str, run_name: str):
    tensorboard_logger = TensorBoardLogger(
        save_dir=log_dir,
        name="PhotonicDARTS_Quant",
        version=run_name,
        default_hp_metric=False,
    )

    print(f"TensorBoard logs will be written to: {tensorboard_logger.log_dir}")

    return Lightning(
        DartsClassificationModule(
            2e-2,
            0.,
            auxiliary_loss_weight=None,
            max_epochs=max_epochs,
            alpha_tracker=alpha_tracker,
        ),
        Trainer(
            accelerator="auto",
            callbacks=[checkpoint_callback],
            logger=tensorboard_logger,
            max_epochs=max_epochs,
        ),
        train_dataloaders=train_loader,
        val_dataloaders=valid_loader,
    )


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/filippo/miniconda3/envs/pytorch-env/lib/python3.11/site-packages/nni/nas/evaluator/pytorch/lightning.py:139: When using training service to spawn trials, please try to wrap PyTorch DataLoader with nni.trace or import DataLoader from nni.nas.evaluator.pytorch.lightning: <torch.utils.data.dataloader.DataLoader object at 0x7fddc03c6810>
/home/filippo/miniconda3/envs/pytorch-env/lib/python3.11/site-packages/nni/nas/evaluator/pytorch/lightning.py:143: When using training service to spawn trials, please try to wrap PyTorch DataLoader with nni.trace or import DataLoader from nni.nas.evaluator.pytorch.lightning: <torch.utils.data.dataloader.DataLoader object at 0x7fddc0363590>


In [36]:
#check CUDA availability and GPU used
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU used: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU used: Quadro RTX 6000


In [ ]:
# Run the quantized DARTS search
# Set num_bits to 6 for 6-bit quantization (can adjust as needed)
experiment_results = search(
    "./tensorboard_logs",
    128,
    num_bits=6,
    run_name="CIFAR10_6bit_search",
)

# The tracker captures in memory during the run; save that history here so it survives the trial process.
print(f"AlphaTracker in notebook currently has {len(alpha_tracker.history)} captured epochs in memory")
if alpha_tracker.history:
    torch.save(alpha_tracker.history, alpha_history_path)
    print(f"Alpha history saved to {alpha_history_path} with {len(alpha_tracker.history)} captured epochs")
else:
    print("AlphaTracker history is empty; nothing to save.")
    print("If this persists, the training hooks are not being reached.")

[2026-06-24 20:37:19] Config is not provided. Will try to infer.
[2026-06-24 20:37:19] Strategy is found to be a one-shot strategy. Setting execution engine to "sequential" and format to "raw".
[2026-06-24 20:37:19] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-06-24 20:37:19] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-06-24 20:37:19] WARNING: `training_service` will be ignored for sequential execution engine.
Initializing LSQ quantizer for 6-bit weight quantization in SafeMixedLinear.
[2026-06-24 20:37:19] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-06-24 20:37:19] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-06-24 20:37:19] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-06-24 20:37:19] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-06-24 20:37:19] WARNING: `trai

/home/filippo/miniconda3/envs/pytorch-env/lib/python3.11/site-packages/nni/nas/evaluator/pytorch/lightning.py:139: When using training service to spawn trials, please try to wrap PyTorch DataLoader with nni.trace or import DataLoader from nni.nas.evaluator.pytorch.lightning: {'train': <torch.utils.data.dataloader.DataLoader object at 0x7ffa501abd10>, 'val': <torch.utils.data.dataloader.DataLoader object at 0x7ffa504b6250>}


[2026-06-24 20:37:19] WARNING: Checkpoint callback does not have last_model_path or best_model_path attribute. Either the strategy has not started, or it did not save any checkpoint: <pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint object at 0x7ffa504ccf50>
[2026-06-24 20:37:19] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-06-24 20:37:19] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-06-24 20:37:19] Checkpoint saved to /home/filippo/TESI_FILIPPO_LUCCHESI/DartsCheckpoints_Quant/wrus0taz/checkpoint.
[2026-06-24 20:37:19] Experiment initialized successfully. Starting exploration strategy...
[2026-06-24 20:37:19] WARNING: Validation dataloaders are missing. Safe to ignore this warning when using one-shot strategy.


You are using a CUDA device ('NVIDIA A2') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/home/filippo/miniconda3/envs/pytorch-env/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/filippo/TESI_FILIPPO_LUCCHESI/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Weight params: 128
Arch params (excluded): 19


...

Trainable params: 1.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.3 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 349                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/filippo/miniconda3/envs/pytorch-env/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

=== Attached alpha gradient hooks ===

Aux exits disabled: running standard forward without exit logits.

Epoch 226:  36%|█████████████████████▍                                      | 112/313 [00:26<00:47,  4.26it/s, v_num=0, train_loss=0.153, train_acc=0.953]

In [ ]:
# Load and visualize alpha evolution throughout the search
import pickle
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import softmax

alpha_history_dir = f'/home/{USER}/TESI_FILIPPO_LUCCHESI/alpha_histories'
alpha_history_pattern = os.path.join(alpha_history_dir, 'alpha_history_6bit*.pt')
alpha_history_candidates = glob.glob(alpha_history_pattern)
if alpha_history_candidates:
    alpha_history_path = max(alpha_history_candidates, key=os.path.getmtime)
    history = torch.load(alpha_history_path, map_location='cpu')
    print(f"Loaded alpha history from {alpha_history_path}: {len(history)} epochs")
else:
    print(f"Warning: no alpha history files found at {alpha_history_pattern}. Run the search cell first.")
    history = None

if history is not None and len(history) > 0:
    # Get all alpha choice names from the first epoch
    alpha_names = sorted(list(history[0]['alphas'].keys()))
    num_alphas = len(alpha_names)
    print(f"Found {num_alphas} architectural choices")
    
    # Create figure with subplots for each alpha
    n_cols = 4
    n_rows = (num_alphas + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3*n_rows))
    if num_alphas == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for idx, name in enumerate(alpha_names):
        ax = axes[idx]
        
        # Extract probabilities for this alpha over all epochs
        probs_list = []
        for epoch_data in history:
            alpha_tensor = epoch_data['alphas'][name]
            probs = softmax(alpha_tensor)
            probs_list.append(probs)
        
        probs_array = np.array(probs_list)  # shape: (num_epochs, num_choices)
        
        # Plot entropy over epochs
        entropy = -np.sum(probs_array * np.log(probs_array + 1e-12), axis=1)
        ax.plot(entropy, linewidth=2, color='steelblue', marker='o', markersize=4)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Entropy')
        ax.set_title(f'{name} - Alpha Polarization', fontsize=10)
        ax.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for idx in range(num_alphas, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'/home/{USER}/TESI_FILIPPO_LUCCHESI/alpha_evolution.png', dpi=150, bbox_inches='tight')
    print(f"Alpha evolution plot saved to /home/{USER}/TESI_FILIPPO_LUCCHESI/alpha_evolution.png")
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*60)
    print("ALPHA POLARIZATION SUMMARY")
    print("="*60)
    for idx, name in enumerate(alpha_names):
        probs_list = []
        for epoch_data in history:
            probs = softmax(epoch_data['alphas'][name])
            probs_list.append(probs)
        probs_array = np.array(probs_list)
        
        # Final entropy
        final_entropy = -np.sum(probs_array[-1] * np.log(probs_array[-1] + 1e-12))
        max_prob_final = np.max(probs_array[-1])
        winning_choice = np.argmax(probs_array[-1])
        
        # Entropy change
        initial_entropy = -np.sum(probs_array[0] * np.log(probs_array[0] + 1e-12))
        entropy_change = initial_entropy - final_entropy
        
        print(f"{name:25s}: entropy={final_entropy:.4f}, max_prob={max_prob_final:.4f}, choice={winning_choice}, Δentropy={entropy_change:.4f}")


In [29]:
# Final supernet state from alpha_history (last captured epoch)
import re
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

alpha_history_dir = '/home/filippo/TESI_FILIPPO_LUCCHESI/alpha_histories'
alpha_history_pattern = os.path.join(alpha_history_dir, 'alpha_history_6bit_CIFAR_v2*.pt')
alpha_history_candidates = glob.glob(alpha_history_pattern)
if not alpha_history_candidates:
    raise FileNotFoundError(f'No alpha history files found at {alpha_history_pattern}. Run the search cell first.')

alpha_history_path = max(alpha_history_candidates, key=os.path.getmtime)
history = torch.load(alpha_history_path, map_location='cpu')
if not history:
    raise RuntimeError('alpha_history is empty; no epochs were captured.')

last_epoch = history[-1]
last_epoch_idx = last_epoch.get('epoch', len(history) - 1)
alphas = last_epoch['alphas']
print(f'Using last captured epoch from alpha_history: {last_epoch_idx} taken from {alpha_history_path} ')
print(f'Choices captured: {len(alphas)}')

def _clean_choice_name(name: str) -> str:
    cleaned = name.replace('._arch_alpha.', '.').replace('_arch_alpha', '')
    cleaned = cleaned.replace('..', '.')
    def _bump_layers(match):
        return f"layers.{int(match.group(1)) + 1}"
    cleaned = re.sub(r"\blayers\.(\d+)\b", _bump_layers, cleaned)
    return cleaned.strip('.')


def _next_versioned_folder(parent_dir: str, base_name: str) -> str:
    os.makedirs(parent_dir, exist_ok=True)
    pattern = re.compile(rf"^{re.escape(base_name)}_v(\d+)$")
    max_v = 0
    for fname in os.listdir(parent_dir):
        match = pattern.match(fname)
        if match:
            max_v = max(max_v, int(match.group(1)))
    folder_path = os.path.join(parent_dir, f"{base_name}_v{max_v + 1}")
    os.makedirs(folder_path, exist_ok=True)
    return folder_path


alpha_summary = {}
for name in sorted(alphas.keys()):
    alpha_tensor = alphas[name].float().numpy()
    exp_vals = np.exp(alpha_tensor - np.max(alpha_tensor))
    probs = exp_vals / np.sum(exp_vals)
    alpha_summary[name] = {
        'raw': alpha_tensor,
        'probs': probs,
        'entropy': -np.sum(probs * np.log(probs + 1e-12)),
        'max_prob': np.max(probs),
        'winning_choice': np.argmax(probs),
        'num_instances': 1,
        'label': _clean_choice_name(name),
    }

alpha_output_root = f'/home/{USER}/TESI_FILIPPO_LUCCHESI/alpha_polarization'
alpha_output_dir = _next_versioned_folder(alpha_output_root, 'alpha_polarization_final_state_6bit')

print(f"Saving vector plots to: {alpha_output_dir}")
print('\n=== FINAL SUPERNET ALPHA STATE (LAST EPOCH) ===')
print(f"{'Choice Name':<45} {'Entropy':<10} {'Max Prob':<10} {'Winner':<7}")
print('-' * 80)
for name in sorted(alpha_summary.keys()):
    data = alpha_summary[name]
    print(f"{data['label']:<45} {data['entropy']:<10.4f} {data['max_prob']:<10.4f} {int(data['winning_choice']):<7d}")

print('\nRaw probabilities per choice (last epoch):')
for name in sorted(alpha_summary.keys()):
    probs_str = np.array2string(alpha_summary[name]['probs'], precision=4, separator=', ')
    print(f"{alpha_summary[name]['label']}: {probs_str}")


def _plot_entropy_bars(labels, entropies, title, file_name):
    fig, ax = plt.subplots(figsize=(10, max(3, 0.45 * max(1, len(labels)))))
    if not entropies:
        ax.set_title(title, fontsize=11)
        ax.text(0.5, 0.5, 'No entries', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
    else:
        ent_min, ent_max = min(entropies), max(entropies)
        normed = (np.array(entropies) - ent_min) / max(ent_max - ent_min, 1e-12)
        colors = plt.cm.RdYlGn(1.0 - normed)
        ax.barh(labels, entropies, color=colors)
        ax.set_xlabel('Entropy (lower = more polarized)', fontsize=10)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='x')
    fig.tight_layout()
    output_path = os.path.join(alpha_output_dir, file_name)
    fig.savefig(output_path, format='svg', bbox_inches='tight')
    plt.close(fig)
    print(f"Saved {output_path}")


# 1. Entropy across 2-choice alphas
_two_choice_labels = []
_two_choice_entropies = []
_three_choice_labels = []
_three_choice_entropies = []
for name in sorted(alpha_summary.keys()):
    label = alpha_summary[name]['label']
    entropy = alpha_summary[name]['entropy']
    num_choices = len(alpha_summary[name]['probs'])
    if num_choices == 2:
        _two_choice_labels.append(label)
        _two_choice_entropies.append(entropy)
    elif num_choices == 3:
        _three_choice_labels.append(label)
        _three_choice_entropies.append(entropy)

_plot_entropy_bars(_two_choice_labels, _two_choice_entropies, 'Entropy (2-choice alphas)', 'entropy_2_choice.svg')
_plot_entropy_bars(_three_choice_labels, _three_choice_entropies, 'Entropy (3-choice alphas)', 'entropy_3_choice.svg')

# 3. Max probability (winning choice confidence)
fig, ax = plt.subplots(figsize=(12, max(3, 0.45 * max(1, len(alpha_summary)))))
choice_names_sorted = [alpha_summary[name]['label'] for name in sorted(alpha_summary.keys())]
indL = 1
indD  = 1
for idx, name in enumerate(choice_names_sorted): 
    if name.startswith("layers"):
        choice_names_sorted[idx] = "Residual Layer " + str(indL)
        indL += 1
    elif name.startswith("downsamples"):
        choice_names_sorted[idx] = "Downsampling " + str(indD)
        indD += 1

max_probs = [alpha_summary[name]['max_prob'] for name in sorted(alpha_summary.keys())]
colors_prob = plt.cm.Greens(np.array(max_probs))
ax.barh(choice_names_sorted, max_probs, color=colors_prob)
ax.set_xlabel('Max Softmax Probability', fontsize=11)
ax.set_title('Confidence in Winning Choice', fontsize=13, fontweight='bold')
ax.set_xlim([0, 1])
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='0.5')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
fig.tight_layout()
output_path = os.path.join(alpha_output_dir, 'max_probability.svg')
fig.savefig(output_path, format='svg', bbox_inches='tight')
plt.close(fig)
print(f"Saved {output_path}")

# 4. Heatmap of softmax probabilities for each choice
fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * max(1, len(choice_names_sorted)))))
heatmap_data = []
max_num_options = 0
for name in sorted(alpha_summary.keys()):
    probs = alpha_summary[name]['probs']
    heatmap_data.append(probs)
    max_num_options = max(max_num_options, len(probs))

# Initialize the numerical array with NaNs so empty options aren't colored
heatmap_array = np.full((len(choice_names_sorted), max_num_options), np.nan)
# Create an annotation array filled with "-"
annot_array = np.full((len(choice_names_sorted), max_num_options), "-", dtype=object)

for i, probs in enumerate(heatmap_data):
    heatmap_array[i, :len(probs)] = probs
    for j, val in enumerate(probs):
        annot_array[i, j] = f"{val:.3f}"

sns.heatmap(
    heatmap_array, 
    annot=annot_array, 
    fmt='s',  # Tells Seaborn to treat the annotations as strings
    cmap='YlOrRd',
    annot_kws={'size': 18}, 
    yticklabels=choice_names_sorted, 
    xticklabels=[f'Option {i}' for i in range(max_num_options)],
    ax=ax, 
    cbar_kws={'label': 'Softmax Probability'}
)

ax.set_title('Softmax Probabilities Heatmap - 6-bit GTSDB, LSQ', fontsize=18, fontweight='bold')
ax.set_xlabel('Architecture Option', fontsize=15)
ax.set_ylabel('Architectural Choice', fontsize=15)
ax.tick_params(axis='y', labelsize=14)
fig.tight_layout()
output_path = os.path.join(alpha_output_dir, 'softmax_heatmap.svg')
fig.savefig(output_path, format='svg', bbox_inches='tight')
plt.close(fig)
print(f"Saved {output_path}")

# 5. Scatter plot: Entropy vs Max Probability
fig, ax = plt.subplots(figsize=(10, 8))
entropies_sorted = [alpha_summary[name]['entropy'] for name in sorted(alpha_summary.keys())]
scatter = ax.scatter(
    entropies_sorted, max_probs, s=300, c=entropies_sorted,
    cmap='RdYlGn_r', alpha=0.7, edgecolors='black', linewidth=1.5
)
for i, name in enumerate(choice_names_sorted):
    ax.annotate(
        name.replace('layer_', 'L').replace('feature', 'F'),
        (entropies_sorted[i], max_probs[i]),
        fontsize=8, ha='center', va='center'
    )
ax.set_xlabel('Entropy (polarization measure)', fontsize=11)
ax.set_ylabel('Max Probability', fontsize=11)
ax.set_title('Entropy - Confidence', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
fig.colorbar(scatter, ax=ax, label='Entropy')
fig.tight_layout()
output_path = os.path.join(alpha_output_dir, 'entropy_confidence_scatter.svg')
fig.savefig(output_path, format='svg', bbox_inches='tight')
plt.close(fig)
print(f"Saved {output_path}")

# 4. Heatmap of softmax probabilities for each choice
fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * max(1, len(choice_names_sorted)))))
heatmap_data = []
max_num_options = 0
for name in sorted(alpha_summary.keys()):
    probs = alpha_summary[name]['probs']
    heatmap_data.append(probs)
    max_num_options = max(max_num_options, len(probs))

# Initialize the numerical array with NaNs so empty options aren't colored
heatmap_array = np.full((len(choice_names_sorted), max_num_options), np.nan)
# Create an annotation array filled with "-"
annot_array = np.full((len(choice_names_sorted), max_num_options), "-", dtype=object)

for i, probs in enumerate(heatmap_data):
    heatmap_array[i, :len(probs)] = probs
    if len(probs) > 0:
        max_val = max(probs)
        for j, val in enumerate(probs):
            if val == max_val:
                annot_array[i, j] = "✓"
            else:
                annot_array[i, j] = "✗"

sns.heatmap(
    heatmap_array, 
    annot=annot_array, 
    fmt='s',  # Tells Seaborn to treat the annotations as strings
    cmap='YlOrRd',
    annot_kws={'size': 18}, 
    yticklabels=choice_names_sorted, 
    xticklabels=[f'Option {i}' for i in range(max_num_options)],
    ax=ax, 
    cbar_kws={'label': 'Softmax Probability'}
)

ax.set_title('Final architecture - 6-bit CIFAR-10, LSQ', fontsize=18, fontweight='bold')
ax.set_xlabel('Architecture Option', fontsize=15)
ax.set_ylabel('Architectural Choice', fontsize=15)
ax.tick_params(axis='y', labelsize=14)
fig.tight_layout()
output_path = os.path.join(alpha_output_dir, 'softmax_heatmap2.svg')
fig.savefig(output_path, format='svg', bbox_inches='tight')
plt.close(fig)
print(f"Saved {output_path}")

Using last captured epoch from alpha_history: 399 taken from /home/filippo/TESI_FILIPPO_LUCCHESI/alpha_histories/alpha_history_6bit_CIFAR_v2.pt 
Choices captured: 10
Saving vector plots to: /home/filippo/TESI_FILIPPO_LUCCHESI/alpha_polarization/alpha_polarization_final_state_6bit_v6

=== FINAL SUPERNET ALPHA STATE (LAST EPOCH) ===
Choice Name                                   Entropy    Max Prob   Winner 
--------------------------------------------------------------------------------
downsamples.1                                 0.4062     0.8593     0      
downsamples.3                                 0.5742     0.7389     1      
downsamples.6                                 0.0360     0.9941     0      
layers.1.op                                   0.1283     0.9764     0      
layers.2.op                                   0.5706     0.7423     1      
layers.3.op                                   0.3566     0.9070     0      
layers.4.op                                   0.2481  

# Results and Best Architecture

In [ ]:
# Export best architecture
best_arch = experiment_results.export_top_models(formatter='instance', top_k=1)[0]
best_arch_desc = experiment_results.export_top_models(formatter='dict', top_k=1)[0]
best_arch_state_dict = best_arch.state_dict()

print("Best Architecture Description:")
print(best_arch_desc)

In [ ]:
# Experiment configuration
print("Experiment Configuration:")
print(experiment_results.config)

# Auto arch code generator

### Load Checkpoint

In [ ]:
checkpoint_path = './checkpoints/best-checkpoint-v31.ckpt'

checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'))

checkpoint_model = CustomDARTSSpace(input_channels=3, channels=64, num_classes=10, layers=7, verbose=0, num_bits=4)

checkpoint_state_dict = checkpoint_model.state_dict()
pretrained_state_dict = checkpoint['state_dict']

if 'global_step' in checkpoint:
    print('Global Step:', checkpoint['global_step'])

if 'callbacks' in checkpoint and isinstance(checkpoint['callbacks'], dict):
    for key, callback in checkpoint['callbacks'].items():
        if isinstance(callback, dict) and 'best_model_score' in callback:
            print('Best Model Score (Train Accuracy):', callback['best_model_score'].item())

In [ ]:
# # Inspect checkpoint contents relevant to supernet vs frozen export
# from collections import Counter

# checkpoint_path = './checkpoints/best-checkpoint.ckpt'
# ckpt = torch.load(checkpoint_path, map_location='cpu')
# sd = ckpt.get('state_dict', {})

# print('Checkpoint top-level keys:', list(ckpt.keys()))
# print('State dict tensor count:', len(sd))

# # Show the first several parameter names so we can see what is actually stored.
# for key in list(sd.keys())[:60]:
#     value = sd[key]
#     shape = tuple(value.shape) if hasattr(value, 'shape') else type(value)
#     print(key, shape)

# # Count the kinds of tensors that appear in the checkpoint.
# category_counts = Counter()
# for key in sd:
#     if 'arch_alpha' in key:
#         category_counts['arch_alpha'] += 1
#     elif 'bn' in key or 'batchnorm' in key.lower():
#         category_counts['batchnorm'] += 1
#     elif '.weight' in key or '.bias' in key:
#         category_counts['weights_or_biases'] += 1
#     else:
#         category_counts['other'] += 1

# print('\nTensor category counts:', dict(category_counts))


In [ ]:
best_arch_desc

# From-Scratch Model

In [ ]:
import json
import os
import re

# Export the best architecture description to JSON for use in other scripts.
def _next_versioned_path(folder: str, base_name: str, ext: str) -> str:
    os.makedirs(folder, exist_ok=True)
    pattern = re.compile(rf"^{re.escape(base_name)}_v(\d+){re.escape(ext)}$")
    max_v = 0
    for fname in os.listdir(folder):
        match = pattern.match(fname)
        if match:
            max_v = max(max_v, int(match.group(1)))
    return os.path.join(folder, f"{base_name}_v{max_v + 1}{ext}")


# Version the choice-index schema. In quant_attention_v1, operation index 2
# means spatial self-attention in layers 5, 6 and 7.
arch_export = dict(best_arch_desc)
arch_export["_search_space_version"] = "quant_attention_v1"
arch_export["_attention_num_heads"] = 8

best_arch_dir = f'/home/{USER}/TESI_FILIPPO_LUCCHESI/best_arch_darts'
arch_export_path = _next_versioned_path(best_arch_dir, 'best_arch_darts_6bit', '.json')
with open(arch_export_path, 'w') as f:
    json.dump(arch_export, f, indent=2)
print(f"Architecture exported to {arch_export_path}")
print("Architecture choices:", arch_export)


In [ ]:
import json
import os
import glob
import re

best_arch_dir = f"/home/{USER}/TESI_FILIPPO_LUCCHESI/best_arch_darts"
pattern = os.path.join(best_arch_dir, "best_arch_darts_6bit_v*.json")
candidates = glob.glob(pattern)
if not candidates:
    raise FileNotFoundError(f"No files found for pattern: {pattern}")


def _architecture_version(path: str) -> int:
    match = re.search(r"_v(\d+)\.json$", os.path.basename(path))
    return int(match.group(1)) if match else -1


arch_path = max(candidates, key=_architecture_version)
print(f"Loading latest architecture file: {arch_path}")

with open(arch_path, 'r') as f:
    arch_dict = json.load(f)

print("Loaded search-space schema:", arch_dict.get("_search_space_version", "unversioned"))
print("Loaded architecture:", arch_dict)


In [ ]:
max_epochs = 400
num_bits = 6


class ActQuant(nn.Module):
    """Unsigned LSQ activation quantizer, matching the search model."""

    def __init__(self, k=6, init_step=0.5, init_alpha=6.0):
        super().__init__()
        self.k = int(k)
        self.step_size = nn.Parameter(torch.tensor(float(init_step)))
        self.alpha = nn.Parameter(torch.tensor(float(init_alpha)))

    def forward(self, x):
        step = torch.clamp(self.step_size, min=1e-8)
        alpha = torch.clamp(self.alpha, min=1e-8)
        qn = 0
        qp = (2 ** self.k) - 1
        grad_scale = 1.0 / ((qp * x.numel()) ** 0.5)
        scaled_step = (step - step.detach()) * grad_scale + step.detach()
        x = torch.clamp(x, -alpha, alpha)
        x_scaled = torch.clamp(x / scaled_step, qn, qp)
        x_rounded = torch.round(x_scaled)
        x_quantized = x_scaled + (x_rounded - x_scaled).detach()
        return x_quantized * scaled_step


class FrozenSignedActQuant(nn.Module):
    """Signed LSQ activation quantizer used inside frozen attention."""

    def __init__(self, k=6, init_step=0.5, init_alpha=6.0):
        super().__init__()
        self.k = int(k)
        self.step_size = nn.Parameter(torch.tensor(float(init_step)))
        self.alpha = nn.Parameter(torch.tensor(float(init_alpha)))

    def forward(self, x):
        step = torch.clamp(self.step_size, min=1e-8)
        alpha = torch.clamp(self.alpha, min=1e-8)
        qn = -(2 ** (self.k - 1))
        qp = 2 ** (self.k - 1) - 1
        grad_scale = 1.0 / ((max(abs(qn), abs(qp)) * x.numel()) ** 0.5)
        scaled_step = (step - step.detach()) * grad_scale + step.detach()
        x = torch.clamp(x, -alpha, alpha)
        x_scaled = torch.clamp(x / scaled_step, qn, qp)
        x_rounded = torch.round(x_scaled)
        x_quantized = x_scaled + (x_rounded - x_scaled).detach()
        return x_quantized * scaled_step


class FrozenUnsignedActQuant(ActQuant):
    """Named unsigned quantizer for attention probabilities."""


from DoReFaLayers import (
    FrozenQuantConv2d,
    FrozenQuantLinear,
    LSQWeightQuantizer,
    dorefa_activation,
)


class FrozenQuantSelfAttention(nn.Module):
    """Fixed-architecture counterpart of QuantMixedSelfAttention.

    It preserves the packed Q/K/V projection, per-tensor LSQ weight
    quantization and every internal activation quantization point used during
    the DARTS search.
    """

    def __init__(self, embed_dim: int, num_heads: int, num_bits: int, dropout: float = 0.0):
        super().__init__()
        if embed_dim % num_heads != 0:
            raise ValueError(
                f"embed_dim={embed_dim} must be divisible by num_heads={num_heads}"
            )
        self.embed_dim = int(embed_dim)
        self.num_heads = int(num_heads)
        self.head_dim = self.embed_dim // self.num_heads
        self.num_bits = int(num_bits)
        self.dropout = float(dropout)

        # Search attention uses bias=False and a packed Q/K/V matrix.
        self.in_proj_weight = nn.Parameter(torch.empty(3 * self.embed_dim, self.embed_dim))
        self.out_proj_weight = nn.Parameter(torch.empty(self.embed_dim, self.embed_dim))
        self._lsq_weight_quantizers = nn.ModuleDict({
            "in_proj": LSQWeightQuantizer(k=self.num_bits, per_channel=False),
            "out_proj": LSQWeightQuantizer(k=self.num_bits, per_channel=False),
        })
        self.act_quants = nn.ModuleDict({
            "query": FrozenSignedActQuant(k=self.num_bits),
            "key": FrozenSignedActQuant(k=self.num_bits),
            "value": FrozenSignedActQuant(k=self.num_bits),
            "q_projected": FrozenSignedActQuant(k=self.num_bits),
            "k_projected": FrozenSignedActQuant(k=self.num_bits),
            "v_projected": FrozenSignedActQuant(k=self.num_bits),
            "scores": FrozenSignedActQuant(k=self.num_bits),
            "probabilities": FrozenUnsignedActQuant(
                k=self.num_bits,
                init_step=1.0 / (2 ** self.num_bits - 1),
                init_alpha=1.0,
            ),
            "context": FrozenSignedActQuant(k=self.num_bits),
            "output": FrozenSignedActQuant(k=self.num_bits),
        })
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.in_proj_weight)
        nn.init.xavier_uniform_(self.out_proj_weight)

    def refresh_weight_quantizers(self):
        self._lsq_weight_quantizers["in_proj"].init_step_size(self.in_proj_weight)
        self._lsq_weight_quantizers["out_proj"].init_step_size(self.out_proj_weight)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        query = self.act_quants["query"](tokens)
        key = self.act_quants["key"](tokens)
        value = self.act_quants["value"](tokens)

        in_proj_weight = self._lsq_weight_quantizers["in_proj"](self.in_proj_weight)
        q_weight, k_weight, v_weight = in_proj_weight.chunk(3, dim=0)
        q = self.act_quants["q_projected"](F.linear(query, q_weight))
        k = self.act_quants["k_projected"](F.linear(key, k_weight))
        v = self.act_quants["v_projected"](F.linear(value, v_weight))

        batch, sequence_length, _ = q.shape
        q = q.reshape(batch, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.reshape(batch, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.reshape(batch, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = self.act_quants["scores"](scores)
        probabilities = torch.softmax(scores, dim=-1)
        probabilities = F.dropout(probabilities, p=self.dropout, training=self.training)
        probabilities = self.act_quants["probabilities"](probabilities)

        context = torch.matmul(probabilities, v)
        context = self.act_quants["context"](context)
        context = context.transpose(1, 2).contiguous().reshape(
            batch, sequence_length, self.embed_dim
        )
        out_proj_weight = self._lsq_weight_quantizers["out_proj"](self.out_proj_weight)
        output = F.linear(context, out_proj_weight)
        return self.act_quants["output"](output)


class FrozenSpatialSelfAttentionBlock(nn.Module):
    """NCHW wrapper matching SpatialSelfAttentionBlock in the search model."""

    def __init__(self, in_channels: int, out_channels: int, num_heads: int, num_bits: int):
        super().__init__()
        self.num_bits = int(num_bits)
        if in_channels == out_channels:
            self.input_projection = nn.Identity()
        else:
            self.input_projection = nn.Sequential(
                FrozenQuantConv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=1,
                    padding=0,
                    bias=False,
                    num_bits=num_bits,
                ),
                nn.BatchNorm2d(out_channels),
                nn.ReLU6(inplace=False),
            )
        self.pre_norm = nn.LayerNorm(out_channels)
        self.attention = FrozenQuantSelfAttention(
            embed_dim=out_channels,
            num_heads=num_heads,
            num_bits=num_bits,
            dropout=0.0,
        )
        self.output_bn = nn.BatchNorm2d(out_channels)
        self.output_relu = nn.ReLU6(inplace=False)

    @staticmethod
    def _run_projection(module, x, w_quant_bits):
        if isinstance(module, nn.Identity):
            return x
        for sublayer in module:
            if isinstance(sublayer, FrozenQuantConv2d):
                x = sublayer(x, w_quant_bits)
            else:
                x = sublayer(x)
        return x

    def forward(self, x: torch.Tensor, w_quant_bits: int | None = None) -> torch.Tensor:
        if w_quant_bits is None:
            w_quant_bits = self.num_bits
        x = self._run_projection(self.input_projection, x, w_quant_bits)
        batch, channels, height, width = x.shape
        tokens = self.pre_norm(x.flatten(2).transpose(1, 2))
        tokens = self.attention(tokens)
        x = tokens.transpose(1, 2).reshape(batch, channels, height, width)
        x = self.output_bn(x)
        return self.output_relu(x)


class FrozenResidualBlock(nn.Module):
    def __init__(self, op: nn.Module, in_channels: int, out_channels: int, num_bits: int):
        super().__init__()
        self.op = op
        if in_channels != out_channels:
            self.proj = nn.Sequential(
                FrozenQuantConv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=1,
                    padding=0,
                    bias=False,
                    num_bits=num_bits,
                ),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.proj = nn.Identity()
        self.post_relu = nn.ReLU6(inplace=False)

    @staticmethod
    def _run_module(module, x, w_quant_bits):
        if isinstance(module, FrozenSpatialSelfAttentionBlock):
            return module(x, w_quant_bits)
        if isinstance(module, nn.Identity):
            return x
        for sublayer in module:
            if isinstance(sublayer, (FrozenQuantConv2d, FrozenQuantLinear)):
                x = sublayer(x, w_quant_bits)
            else:
                x = sublayer(x)
        return x

    def forward(self, x: torch.Tensor, w_quant_bits: int) -> torch.Tensor:
        out = self._run_module(self.op, x, w_quant_bits)
        res = self._run_module(self.proj, x, w_quant_bits)
        return self.post_relu(out + res)


class PhotonicArchQuant(nn.Module):
    def __init__(
        self,
        drop_path_prob=0.0,
        arch_dict=None,
        num_bits=6,
        max_epochs=200,
        exit_points=None,
        early_exit=False,
        exit_thresholds=None,
    ):
        super().__init__()
        if arch_dict is None:
            raise ValueError("arch_dict is required to build PhotonicArchQuant")
        self.arch_dict = arch_dict
        self.search_space_version = arch_dict.get(
            "_search_space_version", "quant_attention_v1"
        )
        self.attention_num_heads = int(arch_dict.get("_attention_num_heads", 8))
        self.drop_path_prob = drop_path_prob
        self.num_bits = int(num_bits)
        self.max_epochs = max_epochs
        self.exit_points = exit_points
        self.early_exit = early_exit
        self.exit_thresholds = exit_thresholds
        in_channels = 32

        self.preliminary_layer = FrozenQuantConv2d(
            3, in_channels, kernel_size=3, padding=1, bias=False, num_bits=num_bits
        )
        self.layer0_bn = nn.BatchNorm2d(in_channels)
        self.layer0_relu = nn.ReLU6(inplace=False)

        default_out_channels = [32, 64, 64, 128, 128, 256, 200]
        layer_out_channels = [
            arch_dict.get(f"layer{i}_out_channels", default_out_channels[i - 1])
            for i in range(1, 8)
        ]

        self.layers = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        self.selected_attention_layers = []
        for layer_index, out_channels in enumerate(layer_out_channels, start=1):
            op_choice = arch_dict.get(
                f"layer_{layer_index}_op", arch_dict.get(f"layer_{layer_index}", 0)
            )
            if isinstance(op_choice, str) and op_choice.isdigit():
                op_choice = int(op_choice)
            if op_choice == 0:
                op = nn.Sequential(
                    FrozenQuantConv2d(
                        in_channels,
                        out_channels,
                        kernel_size=3,
                        stride=1,
                        padding=1,
                        bias=False,
                        num_bits=num_bits,
                    ),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU6(inplace=False),
                )
            elif op_choice == 1:
                op = nn.Sequential(
                    FrozenQuantConv2d(
                        in_channels,
                        in_channels,
                        kernel_size=3,
                        stride=1,
                        padding=1,
                        groups=in_channels,
                        bias=False,
                        num_bits=num_bits,
                    ),
                    nn.BatchNorm2d(in_channels),
                    nn.ReLU6(inplace=False),
                    FrozenQuantConv2d(
                        in_channels,
                        out_channels,
                        kernel_size=1,
                        stride=1,
                        padding=0,
                        bias=False,
                        num_bits=num_bits,
                    ),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU6(inplace=False),
                )
            elif op_choice == 2:
                if self.search_space_version == "quant_attention_v1":
                    if layer_index not in (5, 6, 7):
                        raise ValueError(
                            f"Attention choice is invalid at layer {layer_index}; "
                            "quant_attention_v1 offers it only at layers 5-7."
                        )
                    op = FrozenSpatialSelfAttentionBlock(
                        in_channels=in_channels,
                        out_channels=out_channels,
                        num_heads=self.attention_num_heads,
                        num_bits=num_bits,
                    )
                    self.selected_attention_layers.append(layer_index)
                elif self.search_space_version == "skipconnect_v1":
                    op = nn.Sequential(SkipConnect(in_channels, out_channels, num_bits=num_bits))
                else:
                    raise ValueError(
                        f"Unknown search-space schema {self.search_space_version!r} "
                        "for operation index 2."
                    )
            else:
                raise ValueError(
                    f"Unsupported operation choice {op_choice!r} at layer {layer_index}"
                )

            self.layers.append(FrozenResidualBlock(op, in_channels, out_channels, num_bits))

            downsample_choice = None
            if layer_index in (2, 4, 7):
                downsample_choice = arch_dict.get(f"layer_{layer_index}_down", 0)
            if downsample_choice == 0:
                self.downsamples.append(nn.AvgPool2d(kernel_size=2, stride=2))
            elif downsample_choice == 1:
                self.downsamples.append(
                    nn.Sequential(
                        FrozenQuantConv2d(
                            out_channels,
                            out_channels,
                            kernel_size=1,
                            stride=2,
                            padding=0,
                            bias=False,
                            num_bits=num_bits,
                        ),
                        nn.BatchNorm2d(out_channels),
                    )
                )
            else:
                self.downsamples.append(nn.Identity())
            in_channels = out_channels

        print("Selected frozen attention layers:", self.selected_attention_layers)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = FrozenQuantLinear(layer_out_channels[-1], 10, num_bits=num_bits)
        self.dropout = nn.Dropout(0.2)
        self.actQuantPre = ActQuant(k=num_bits)
        self.actQuantPost = ActQuant(k=num_bits)
        self.actQuants = nn.ModuleList(
            [ActQuant(k=num_bits) for _ in range(len(layer_out_channels))]
        )

        if exit_points is None:
            exit_points = [3, 5, 7]
        self.exit_points = sorted(set(exit_points))
        self.exit_heads = nn.ModuleDict()
        for layer_index in self.exit_points:
            if 1 <= layer_index <= len(layer_out_channels):
                out_channels = layer_out_channels[layer_index - 1]
                self.exit_heads[str(layer_index)] = self._make_exit_head(out_channels, num_bits)

    def _make_exit_head(self, in_channels: int, num_bits: int) -> nn.Module:
        return nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            FrozenQuantLinear(in_channels, 10, num_bits=num_bits),
        )

    @staticmethod
    def _run_quantized_sequence(module, x, w_quant_bits):
        if isinstance(module, nn.Identity):
            return x
        for sublayer in module:
            if isinstance(sublayer, (FrozenQuantConv2d, FrozenQuantLinear)):
                x = sublayer(x, w_quant_bits)
            else:
                x = sublayer(x)
        return x

    def _exit_confidence(self, logits: torch.Tensor) -> torch.Tensor:
        return torch.softmax(logits, dim=1).max(dim=1).values

    def _forward_core(
        self,
        x,
        act_quant_bits,
        w_quant_bits,
        pre_act_bits=None,
        post_act_bits=None,
        early_exit=False,
        exit_thresholds=None,
        return_all_exits=False,
    ):
        if pre_act_bits is not None:
            x = self.actQuantPre(x)
        x = self.preliminary_layer(x, w_quant_bits)
        x = self.layer0_bn(x)
        x = self.layer0_relu(x)

        exit_logits = []
        for i, block in enumerate(self.layers, start=1):
            # Match the NAS supernet: quantize the block input, then execute
            # the selected operation plus its outer residual connection.
            x = self.actQuants[i - 1](x)
            x = block(x, w_quant_bits)

            downsample = self.downsamples[i - 1]
            if isinstance(downsample, nn.Sequential):
                x = self._run_quantized_sequence(downsample, x, w_quant_bits)
            elif not isinstance(downsample, nn.Identity):
                x = downsample(x)

            if str(i) in self.exit_heads:
                head = self.exit_heads[str(i)]
                head_logits = self._run_quantized_sequence(head, x, w_quant_bits)
                exit_logits.append((i, head_logits))
                if early_exit and not self.training:
                    threshold = None
                    if isinstance(exit_thresholds, dict):
                        threshold = exit_thresholds.get(i)
                    elif isinstance(exit_thresholds, (list, tuple)) and len(exit_thresholds) >= len(exit_logits):
                        threshold = exit_thresholds[len(exit_logits) - 1]
                    if threshold is not None:
                        confidence = self._exit_confidence(head_logits)
                        if torch.all(confidence >= float(threshold)):
                            print(
                                f"Early exiting at layer {i} with confidence "
                                f"{confidence.min().item():.4f} (threshold: {threshold})"
                            )
                            return head_logits

        x = self.pool(x)
        x = torch.flatten(x, 1)
        if post_act_bits is not None:
            x = self.actQuantPost(x)
        x = self.classifier(x, w_quant_bits)
        if return_all_exits:
            exit_logits.append((len(self.layers), x))
            return x, exit_logits
        if early_exit and not self.training:
            return x, len(self.layers)
        return x

    def forward_train(self, x, epoch: int = 0, return_all_exits: bool = False):
        return self._forward_core(
            x,
            self.num_bits,
            self.num_bits,
            pre_act_bits=self.num_bits,
            post_act_bits=self.num_bits,
            return_all_exits=return_all_exits,
        )

    def forward_infer(
        self,
        x,
        num_bits: int | None = None,
        early_exit: bool = False,
        exit_thresholds=None,
        return_all_exits: bool = False,
    ):
        if num_bits is None:
            num_bits = self.num_bits
        return self._forward_core(
            x,
            num_bits,
            num_bits,
            pre_act_bits=num_bits,
            post_act_bits=num_bits,
            early_exit=early_exit,
            exit_thresholds=exit_thresholds,
            return_all_exits=return_all_exits,
        )

    def forward(
        self,
        x,
        epoch: int | None = None,
        inference: bool = False,
        return_all_exits: bool = False,
    ):
        if epoch is None or epoch == 0:
            epoch = getattr(self, "epoch", epoch)
        self.epoch = 0 if epoch is None else epoch
        if inference or not self.training:
            return self.forward_infer(
                x,
                early_exit=self.early_exit,
                exit_thresholds=self.exit_thresholds,
                return_all_exits=return_all_exits,
            )
        return self.forward_train(x, self.epoch, return_all_exits=return_all_exits)

    def init_weights(self):
        for module in self.modules():
            if isinstance(module, FrozenQuantSelfAttention):
                module.reset_parameters()
            elif isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

        # LSQ steps must reflect the weights after initialization. This is
        # done before the optimizer is created, so replacing the parameters is safe.
        for module in self.modules():
            if isinstance(module, (FrozenQuantConv2d, FrozenQuantLinear)):
                module.weight_quantizer.init_step_size(module.weight)
            elif isinstance(module, FrozenQuantSelfAttention):
                module.refresh_weight_quantizers()

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob


In [ ]:
exit_thresholds = {
    3: 0.90,
    5: 0.95,
    7: 0.98,
}
scratch_model = PhotonicArchQuant(arch_dict=arch_dict, num_bits=num_bits, early_exit=False, max_epochs=max_epochs, exit_thresholds=None)
scratch_model
for m in scratch_model.modules():
    if isinstance(m, ActQuant):
        m.step_size.data.fill_(0.05)

In [ ]:
# Sanity check: early-exit outputs and auxiliary-loss format
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
scratch_model.to(device)
scratch_model.eval()

with torch.no_grad():
    inputs, labels = next(iter(train_loader))
    inputs, labels = inputs.to(device), labels.to(device)
    final_logits, exits = scratch_model(inputs, inference=True, return_all_exits=True)
    print("Final logits shape:", tuple(final_logits.shape))
    print("Exit heads:", [layer for layer, _ in exits])
    print("Exit shapes:", [tuple(logits.shape) for _, logits in exits])

### Rename dictionary keys

In [ ]:
best_arch_state_dict = scratch_model.state_dict()

## Load weights

In [ ]:
scratch_model.load_state_dict(best_arch_state_dict)

In [ ]:
def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total_samples
    return avg_loss, accuracy

In [ ]:
from pytorch_lightning.callbacks import Callback

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class StopOnValAccDrop(Callback):
    def __init__(self, drop_tolerance: float = 0.02, metric_name: str = "val_acc"):
        super().__init__()
        self.drop_tolerance = drop_tolerance
        self.metric_name = metric_name
        self.best = None

    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        if self.metric_name not in metrics:
            return
        current = metrics[self.metric_name]
        current_val = float(current.detach().cpu()) if hasattr(current, "detach") else float(current)
        if self.best is None or current_val > self.best:
            self.best = current_val
            return
        if (current_val < self.best - self.drop_tolerance) and (current_val > 0.88):
            print(
                f"Early stopping: {self.metric_name} dropped from {self.best:.4%} to {current_val:.4%} "
                f"(tolerance {self.drop_tolerance:.2%})."
            )
            trainer.should_stop = True


scratch_model.to(device)
scratch_model.init_weights()
scratch_model.train()
module = DartsClassificationModule(2e-2, 0., auxiliary_loss_weight=0.4, max_epochs=max_epochs)
module.set_model(scratch_model)


val_acc_drop_tolerance = 0.02
early_stop_callback = StopOnValAccDrop(
    drop_tolerance=val_acc_drop_tolerance,
    metric_name="val_acc",
)
trainer = Trainer(
    max_epochs=module.max_epochs,
    accelerator="auto",
    gradient_clip_val=1.0,
    callbacks=[early_stop_callback],
)

In [ ]:
# Auxiliary-exit demo: pass weights and compute main/aux losses on one batch


inputs, labels = next(iter(train_loader))
inputs, labels = inputs.to(device), labels.to(device)
model = getattr(module, "_model", getattr(module, "model", None))
if model is None:
    raise RuntimeError("DartsClassificationModule has no attached model.")
model.train()
outputs = model(inputs, inference=True, return_all_exits=True)
y_hat, exit_logits = module._resolve_outputs(outputs)
loss_main = module.criterion(y_hat, labels)
loss_aux = module._compute_aux_exit_loss(exit_logits, labels)
print("loss_main:", float(loss_main), "loss_aux:", None if loss_aux is None else float(loss_aux))

In [ ]:
# Get the current Jupyter kernel process
p = psutil.Process(os.getpid())

# Set the niceness to 19 (Lowest CPU priority)
p.nice(19)

aux_weights = [0.2] * max(1, len(getattr(scratch_model, "exit_points", [])))
module = DartsClassificationModule(2e-2, 0.0, 0.4, max_epochs, auxiliary_exit_weights=aux_weights)
module.set_model(scratch_model)
scratch_model.early_exit = False
module.train()

trainer.fit(module, train_loader, valid_loader)

In [ ]:
# Tune early-exit confidence thresholds on validation data
import time

thresholds = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0]
max_batches = None  # set to None to use the full validation set
model = scratch_model
model.to(device)
model.eval()

exit_points = getattr(model, "exit_points", [])
if not exit_points:
    raise RuntimeError("No exit points found on the model.")

single_loader = DataLoader(
    test_data,
    batch_size=1,
    pin_memory=True,
    num_workers=0,
    persistent_workers=False,
    shuffle=False,
    drop_last=False,
 )

def _choose_exit_layer(exits, thresholds_by_layer):
    chosen_layer = exits[-1][0]
    chosen_logits = exits[-1][1]
    for layer_idx, head_logits in exits:
        threshold = thresholds_by_layer.get(int(layer_idx))
        if threshold is None:
            continue
        confidence = torch.softmax(head_logits, dim=1).max().item()
        if confidence >= float(threshold):
            return layer_idx, head_logits
    return chosen_layer, chosen_logits

with torch.no_grad():
    for thr in thresholds:
        start_time = time.perf_counter()
        thr_dict = {int(p): float(thr) for p in exit_points}
        total = 0
        correct = 0
        exit_layer_sum = 0
        used_batches = 0
        for inputs, labels in single_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            final_logits, exits = model(
                inputs,
                inference=True,
                return_all_exits=True,
            )
            exit_layer, logits = _choose_exit_layer(exits, thr_dict)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            exit_layer_sum += int(exit_layer) * labels.size(0)
            used_batches += 1
            if max_batches is not None and used_batches >= max_batches:
                break
        acc = correct / max(1, total)
        avg_exit = exit_layer_sum / max(1, total)
        elapsed = time.perf_counter() - start_time
        print(f"thr={thr:.2f}  acc={acc:.4%}  avg_exit_layer={avg_exit:.2f}  time={elapsed:.2f}s")

In [ ]:
criterion = nn.CrossEntropyLoss()
scratch_model.to(device)

valid_loader = DataLoader(
    test_data, batch_size=batch_size,
    pin_memory=True, num_workers=0, persistent_workers=False
)

scratch_model.eval()
total_loss = 0.0
correct = 0
total_samples = 0
correct_flags = []

with torch.no_grad():
    for inputs, labels in valid_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = scratch_model(inputs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        batch_correct = preds.eq(labels).detach().cpu().numpy().astype(np.float32)
        correct_flags.extend(batch_correct.tolist())
        correct += int(batch_correct.sum())
        total_samples += labels.size(0)

avg_loss = total_loss / max(1, len(valid_loader))
accuracy = correct / max(1, total_samples)

if correct_flags:
    correct_arr = np.array(correct_flags, dtype=np.float32)
    n_boot = 1000
    rng = np.random.default_rng(0)
    boot_accs = np.empty(n_boot, dtype=np.float32)
    n = len(correct_arr)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_accs[i] = correct_arr[idx].mean()
    max_acc = float(boot_accs.max())
    min_acc = float(boot_accs.min())
    ci_low, ci_high = np.percentile(boot_accs, [2.5, 97.5]).tolist()
else:
    max_acc = float("nan")
    min_acc = float("nan")
    ci_low = float("nan")
    ci_high = float("nan")

print(f"Validation Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4%}")
print(f"Max val acc (bootstrap): {max_acc:.4%}")
print(f"Min val acc (bootstrap): {min_acc:.4%}")
print(f"95% CI (bootstrap acc): [{ci_low:.4%}, {ci_high:.4%}]")

In [ ]:
# Export the final scratch model for use in other scripts
model_folder = f'/home/{USER}/TESI_FILIPPO_LUCCHESI/scratch_models'
final_model_path = _next_versioned_path(model_folder, 'photonic_arch_quant_6bit', '.pth')
torch.save(scratch_model.state_dict(), final_model_path)
print(f"Final model state dict saved to {final_model_path}")

In [ ]:
# Create missing LSQWeightQuantizer.step_size params from saved state and copy values
import torch, glob, os

saved_path = f'/home/{USER}/TESI_FILIPPO_LUCCHESI/scratch_models/photonic_arch_quant_6bit_v2.pth'
saved = torch.load(saved_path, map_location='cpu')

# instantiate model as you did
m = PhotonicArchQuant(arch_dict=arch_dict, num_bits=num_bits, early_exit=False, max_epochs=max_epochs, exit_thresholds=None)

# For each saved key that endswith step_size, ensure the parent module has that attribute as nn.Parameter
created = []
for k, v in saved.items():
    if k.endswith('.step_size'):
        parts = k.split('.')
        parent_parts = parts[:-1]
        obj = m
        ok = True
        for p in parent_parts:
            if hasattr(obj, p):
                obj = getattr(obj, p)
            elif p.isdigit():
                obj = obj[int(p)]
            else:
                ok = False
                break
        if not ok:
            print('Parent not found for', k); continue
        final = parts[-1]
        if hasattr(obj, final):
            # already exists, will copy below
            continue
        # create Parameter from saved tensor (clone to avoid shared storage)
        tensor = v.clone().detach()
        param = torch.nn.Parameter(tensor)
        setattr(obj, final, param)
        created.append(k)

print('Created step_size params for', len(created), 'entries')
# Now load state dict strictly to copy remaining values
info = m.load_state_dict(saved, strict=False)
print('After load_state_dict(strict=False):', 'missing=', len(info.missing_keys), 'unexpected=', len(info.unexpected_keys))

# Verify step_size values copied
for k in saved.keys():
    if k.endswith('.step_size'):
        parts = k.split('.')
        obj = m
        for p in parts[:-1]:
            obj = getattr(obj, p) if hasattr(obj, p) else obj[int(p)]
        val = getattr(obj, parts[-1])
        print(k, '->', float(val.data.mean()))

In [ ]:
# 1) Mark LSQ quantizers initialized and verify step_size matches saved values
import torch

saved = torch.load(f'/home/{USER}/TESI_FILIPPO_LUCCHESI/scratch_models/photonic_arch_quant_6bit_v2.pth', map_location='cpu')

# mark initialized for each LSQWeightQuantizer we found previously
from DoReFaLayers import LSQWeightQuantizer

m.eval()
for name, mod in m.named_modules():
    if isinstance(mod, LSQWeightQuantizer):
        if hasattr(mod, 'step_size'):
            mod.initialized = True
        else:
            print('No step_size on', name)

# 2) Quick verification: compare saved vs model step_size values where possible
def get_saved_tensor(key):
    return saved.get(key, None)

mismatch = []
for k in saved.keys():
    if k.endswith('.step_size'):
        parts = k.split('.')
        obj = m
        found = True
        for p in parts[:-1]:
            if hasattr(obj, p):
                obj = getattr(obj, p)
            elif p.isdigit():
                obj = obj[int(p)]
            else:
                found = False
                break
        if not found:
            mismatch.append((k, 'parent_missing'))
            continue
        final = parts[-1]
        if not hasattr(obj, final):
            mismatch.append((k, 'attr_missing'))
            continue
        model_val = getattr(obj, final)
        saved_val = get_saved_tensor(k)
        if model_val.shape != saved_val.shape:
            mismatch.append((k, f'shape_mismatch model={tuple(model_val.shape)} saved={tuple(saved_val.shape)}'))
            continue
        diff = float((model_val.detach().cpu() - saved_val.detach().cpu()).abs().mean())
        if diff > 1e-6:
            mismatch.append((k, f'mean_abs_diff={diff:.3e}'))

print('Verification mismatches (empty = good):')
for it in mismatch[:50]:
    print(' ', it)

# 3) Run evaluation (make sure model on correct device)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
m.to(device)
m.eval()

# (use your existing evaluate_model function or run a quick sanity batch)
# Example quick forward on one batch:
inputs, labels = next(iter(valid_loader))
inputs = inputs.to(device)
with torch.no_grad():
    out = m(inputs)
print('Sample forward OK, output shape:', tuple(out.shape))

In [ ]:
criterion = nn.CrossEntropyLoss()
m.to(device)

valid_loader = DataLoader(
    test_data, batch_size=batch_size,
    pin_memory=True, num_workers=0, persistent_workers=False
)

m.eval()
total_loss = 0.0
correct = 0
total_samples = 0
correct_flags = []

with torch.no_grad():
    for inputs, labels in valid_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = m(inputs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        batch_correct = preds.eq(labels).detach().cpu().numpy().astype(np.float32)
        correct_flags.extend(batch_correct.tolist())
        correct += int(batch_correct.sum())
        total_samples += labels.size(0)

avg_loss = total_loss / max(1, len(valid_loader))
accuracy = correct / max(1, total_samples)

if correct_flags:
    correct_arr = np.array(correct_flags, dtype=np.float32)
    n_boot = 1000
    rng = np.random.default_rng(0)
    boot_accs = np.empty(n_boot, dtype=np.float32)
    n = len(correct_arr)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_accs[i] = correct_arr[idx].mean()
    max_acc = float(boot_accs.max())
    min_acc = float(boot_accs.min())
    ci_low, ci_high = np.percentile(boot_accs, [2.5, 97.5]).tolist()
else:
    max_acc = float("nan")
    min_acc = float("nan")
    ci_low = float("nan")
    ci_high = float("nan")

print(f"Validation Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4%}")
print(f"Max val acc (bootstrap): {max_acc:.4%}")
print(f"Min val acc (bootstrap): {min_acc:.4%}")
print(f"95% CI (bootstrap acc): [{ci_low:.4%}, {ci_high:.4%}]")

In [ ]:
# Load and inspect the latest photonic_arch_quant_6bit model
import os
import glob

# Find the latest photonic_arch_quant_6bit model
model_folder = f'/home/{USER}/TESI_FILIPPO_LUCCHESI/scratch_models'
model_pattern = os.path.join(model_folder, 'photonic_arch_quant_6bit*.pth')
model_files = sorted(glob.glob(model_pattern))

if not model_files:
    print(f"No photonic_arch_quant_6bit models found in {model_folder}")
else:
    latest_model_path = model_files[-1]  # Get the latest (highest version)
    print(f"Loading latest model: {latest_model_path}")
    
    # Load the state dict
    state_dict = torch.load(latest_model_path, map_location='cpu')
    print(f"\nModel state dict loaded with {len(state_dict)} parameters")
    
    # Inspect state dict keys to understand the model structure
    print("\n" + "="*70)
    print("STATE DICT ANALYSIS")
    print("="*70)
    
    # Check for quantizer parameters (indicates LSQ or similar)
    quantizer_keys = [k for k in state_dict.keys() if 'quantizer' in k]
    if quantizer_keys:
        print(f"\nFound {len(quantizer_keys)} quantizer-related parameters (LSQ quantizers detected)")
        print("Sample quantizer keys:")
        for key in quantizer_keys[:5]:
            print(f"  - {key}")
    
    # Create PhotonicArchQuant instance with correct parameters
    print("\n" + "="*70)
    print("INSTANTIATING PhotonicArchQuant")
    print("="*70)
    
    print(f"\nUsing parameters:")
    print(f"  arch_dict: {type(arch_dict).__name__} with {len(arch_dict)} keys")
    print(f"  num_bits: {num_bits}")
    print(f"  max_epochs: {max_epochs}")
    print(f"  early_exit: True")
    print(f"  exit_thresholds: {exit_thresholds}")
    
    # Create the correct model instance (PhotonicArchQuant)
    model = PhotonicArchQuant(arch_dict=arch_dict, num_bits=num_bits, early_exit=False, max_epochs=max_epochs, exit_thresholds=exit_thresholds)
    
    # Load the state dict with strict=False to handle quantizer parameter mismatches
    print(f"\nLoading state dict with strict=False to handle quantizer parameters...")
    incompatible_keys = model.load_state_dict(state_dict, strict=False)
    print(f"State dict loaded successfully (non-strict mode)")
    print(f"  Missing keys: {len(incompatible_keys.missing_keys)}")
    print(f"  Unexpected keys: {len(incompatible_keys.unexpected_keys)}")
    
    if incompatible_keys.unexpected_keys:
        print(f"\nUnexpected keys (will be ignored - these are old quantizer parameters):")
        for key in incompatible_keys.unexpected_keys[:10]:
            print(f"  - {key}")
        if len(incompatible_keys.unexpected_keys) > 10:
            print(f"  ... and {len(incompatible_keys.unexpected_keys) - 10} more")
    
    # Set model to eval mode
    model.eval()
    print(f"\nModel set to evaluation mode")
    
    # ===== INSPECT MODEL ARCHITECTURE =====
    print("\n" + "="*70)
    print("MODEL ARCHITECTURE INSPECTION")
    print("="*70)
    
    # Print model summary
    print(f"\nModel Type: {type(model).__name__}")
    print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    # ===== EXTRACT AND PRINT CONVOLUTION OUTPUT CHANNELS =====
    print("\n" + "="*70)
    print("CONVOLUTION OUTPUT CHANNELS")
    print("="*70)
    
    conv_info = []
    conv_counter = 0
    
    for name, module in model.named_modules():
        # Check for Conv2d layers (both standard and quantized versions)
        if isinstance(module, nn.Conv2d):
            out_channels = module.out_channels
            in_channels = module.in_channels
            kernel_size = module.kernel_size
            stride = module.stride
            padding = module.padding
            groups = getattr(module, 'groups', 1)
            
            conv_counter += 1
            info = {
                'id': conv_counter,
                'name': name,
                'module_type': type(module).__name__,
                'in_channels': in_channels,
                'out_channels': out_channels,
                'kernel_size': kernel_size,
                'stride': stride,
                'padding': padding,
                'groups': groups,
            }
            conv_info.append(info)
    
    # Print table
    print(f"\nTotal Convolutions Found: {len(conv_info)}\n")
    print(f"{'ID':<4} {'Layer Name':<50} {'Type':<20} {'In Ch':<8} {'Out Ch':<8} {'K':<5} {'S':<3} {'G':<3}")
    print("-" * 120)
    
    for info in conv_info:
        kernel_str = f"{info['kernel_size'][0]}x{info['kernel_size'][1]}" if isinstance(info['kernel_size'], tuple) else str(info['kernel_size'])
        stride_str = f"{info['stride'][0]}" if isinstance(info['stride'], tuple) else str(info['stride'])
        print(f"{info['id']:<4} {info['name']:<50} {info['module_type']:<20} {info['in_channels']:<8} {info['out_channels']:<8} {kernel_str:<5} {stride_str:<3} {info['groups']:<3}")
    
    # Summary statistics
    print("\n" + "="*70)
    print("CHANNEL STATISTICS")
    print("="*70)
    out_channels_list = [info['out_channels'] for info in conv_info]
    print(f"Output Channels - Min: {min(out_channels_list)}, Max: {max(out_channels_list)}, Mean: {np.mean(out_channels_list):.1f}")
    print(f"Total unique output channel values: {len(set(out_channels_list))}")
    
    # Show distribution
    from collections import Counter
    channel_dist = Counter(out_channels_list)
    print(f"\nOutput Channel Distribution:")
    for channels, count in sorted(channel_dist.items()):
        print(f"  {channels} channels: {count} convolutions")
